# Spark-X2.6-4B-En4B-A1B (`SparkX-Loop v0.3`) — One-Shot Unattended Colab G4 Training Notebook
### **8.2B Total / 5.2B Active Parameters | 72 Effective Layers | 3-Tier Zipfian MoE-Engram (`4.0B` Total / `1.0B` Active) | Runtime $O(n \log n)$ Log-Linear KV Cache**

This self-contained notebook builds, warm-starts, trains, evaluates, and exports **`Spark-X2.6-4B-En4B-A1B` (`SparkX-Loop v0.3`)** in a single unattended **Run All** session on Google Colab (`G4 96 GB` / `RTX PRO 6000 Blackwell` / `A100` / `L4`).

---

### Architectural Summary (`Spark-X2.6-4B-En4B-A1B`)

| Subsystem | Total Parameters | Active / Pass | Placement & Role |
| :--- | :---: | :---: | :--- |
| **Looped `Spark-X2.6-4B` Backbone (36 Physical $\rightarrow$ 72 Virtual Layers)** | `4.20B` | `4.20B` (`100%`) | **GPU HBM (`BF16`)**: 3-Pass Telescoping Recurrence (`[0..4)` Prelude + Pass 1 `[4..28)` + Pass 2 `[4..28)` + Pass 3 `[12..24)` + `[28..36)` Coda) |
| **Sublayer Dual-mHC Controllers + Pass-0/1/2 Adapters** | `0.025B` | `0.025B` (`100%`) | **GPU HBM (`FP32`)**: Closed-form $2\times 2$ Birkhoff doubly-stochastic routing + zero-gated Pass-3 (`p3_extra_gain = 0`) |
| **Tier 1A: Hot Lexical N-Gram Cache (8 Anchors $\times$ `16,384` $\times$ 3 scales)** | `0.201B` | $<0.001B$ ($O(1)$) | **GPU HBM (`BF16`)**: Zero-latency deterministic N-gram hash lookup for top $\sim 85\%$ Zipfian token patterns |
| **Tier 1B: Cold Long-Tail N-Gram Bank (8 Anchors $\times$ `49,152` $\times$ 3 scales)** | `0.604B` | $<0.001B$ ($O(1)$) | **Pinned Host DRAM (`130 GB` CPU RAM)**: `0 GB` GPU VRAM; ahead-of-time prefetch + `<0.3 ms/step` coalesced sparse row Adam |
| **Tier 2: Hot Shared SwiGLU Expert (8 Anchors, `2560 -> 5888 -> 2560`)** | `0.362B` | `0.362B` (`100%`) | **GPU HBM (`BF16`)**: Always-active shared factual & syntactic memory expert |
| **Tier 3: Routed SwiGLU Memory Experts (8 Anchors $\times$ `E=32, Top-K=7, rank=3072`)** | `2.819B` | `0.617B` (`21.9%`) | **GPU HBM (`BF16`)**: Coarse-to-Fine Upcycled (`8 -> 32`) experts with X-MoE Cosine + Sigmoid + Z-Loss routing |
| **Context Gates & Zero-Init Residual Projections (8 Anchors)** | `0.034B` | `0.034B` (`100%`) | **GPU HBM (`BF16`)**: Guarantees exact `max_abs_diff == 0.0` at Step 0 |
| **Grand Total (`Spark-X2.6-4B-En4B-A1B`)** | **`8.245B` Total** | **`5.238B` Active** | **Fits comfortably inside a single Colab GPU with full `BF16` backbone + `4.02B` Engram!** |

---

### 4-Stage Progressive Depth & Expert Upcycling Curriculum
1. **Stage 1 (`warmup` — 36 Virtual Layers, `8` Coarse Macro-Experts, `Top-K = 4`)**: Freezes the warm-started `v0.2` backbone for 10% of steps while warming up Tier-2 Shared Experts, the 8 Coarse Macro-Experts per anchor, and Tier-1A/1B N-gram tables.
2. **Stage 2 (`R12` — 48 Virtual Layers, `8` Coarse Macro-Experts, `Top-K = 4`)**: Unfreezes the full 36-layer backbone (`lr = 2e-6`) with 2-pass recurrence on `[8..20)`.
3. **Stage 3 (`R24` — 60 Virtual Layers, Upcycles `8 -> 32` Fine-Grained Experts, `Top-K = 7`)**: Calls `model.upcycle_all_engrams_8_to_32()` to clone and orthogonally perturb the 8 converged macro-experts into 32 specialized experts (`1.01B` active Engram) across `[4..28)`.
4. **Stage 4 (`R72` — 72 Virtual Layers, `32` Experts, `Top-K = 7`)**: Activates Pass-3 deep inner-core recurrence over `[12..24)` (`enable_p3 = True`, guarded by zero-initialized `tanh(p3_extra_gain)`).


In [ ]:
# Cell 2: Unattended Environment Bootstrap (Pinned transformers==4.57.1 + bitsandbytes + Unsloth Zoo)
import importlib.metadata
import os
import subprocess
import sys

REQUIRED_TRANSFORMERS = '4.57.1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


def _pkg_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except Exception:
        return ''


preloaded_tf = sys.modules.get('transformers')
preloaded_tf_ver = getattr(preloaded_tf, '__version__', None) if preloaded_tf else None

need_install = (
    _pkg_version('transformers') != REQUIRED_TRANSFORMERS
    or not _pkg_version('bitsandbytes')
    or not _pkg_version('peft')
    or not _pkg_version('datasets')
)

if need_install:
    print('Installing pinned dependencies for Spark-X2.6-4B-En4B-A1B (v0.3)...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'transformers=={REQUIRED_TRANSFORMERS}',
        'accelerate>=1.2.0',
        'bitsandbytes>=0.45.0',
        'datasets>=3.2.0',
        'huggingface_hub>=0.28.0',
        'peft>=0.14.0',
        'safetensors>=0.4.5',
        'sentencepiece',
        'einops',
    ])
    for optional_pkg in ('cut-cross-entropy', 'unsloth_zoo', 'unsloth'):
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', optional_pkg],
            check=False,
        )

    if preloaded_tf_ver is not None and preloaded_tf_ver != REQUIRED_TRANSFORMERS:
        print(f'Restarting Colab kernel once to switch in-memory transformers {preloaded_tf_ver} -> {REQUIRED_TRANSFORMERS}...')
        os.kill(os.getpid(), 9)

import torch
import transformers
print(f'PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | CUDA available: {torch.cuda.is_available()}')


In [ ]:
# Cell 3: Central Configuration for Spark-X2.6-4B-En4B-A1B (SparkX-Loop v0.3)
CFG = {
    # Repositories
    'BASE_MODEL_ID': 'XHToken/Spark-X2.6-4B',
    'WARM_START_V02_REPO': 'TassP0lar/Spark-X2.6-4B-Loop-mHC-R24',  # Set to '' to train from raw Spark-X2.6-4B
    'ENABLE_V02_WARM_START': True,
    'HF_HUB_REPO_ID': 'TassP0lar/Spark-X2.6-4B-En4B-A1B',
    'PUSH_TO_HUB': True,
    'PRIVATE_HUB_REPO': False,
    'OUTPUT_DIR': './checkpoints/sparkx_v03_en4b_a1b',
    'MAX_SAVED_CHECKPOINTS': 3,

    # v0.3 Architecture Settings (8.2B Total / 5.2B Active, 72 Effective Layers)
    'LOOP_START': 4,
    'LOOP_END': 28,
    'LOOP_PASSES': 2,
    'P3_START': 12,
    'P3_END': 24,
    'MAX_PASSES': 4,
    'MHC_STREAMS': 2,
    'PASS_ADAPTER_RANK': 16,
    'BACKBONE_ENGRAM_STRIDE': 4,     # 8 Macro-Block Anchors: Layers 3, 7, 11, 15, 19, 23, 27, 31

    # 3-Tier Zipfian 4.0B-A1B MoE-Engram Settings (per anchor across 8 anchors)
    'ENG_HOT_BUCKETS': 16384,        # Tier 1A (GPU HBM hot cache): 25.17M/anchor (0.201B total)
    'ENG_COLD_BUCKETS': 49152,       # Tier 1B (Pinned Host DRAM cold bank): 75.50M/anchor (0.604B total, 0 GB VRAM)
    'ENG_DIM': 512,
    'ENG_SHARED_RANK': 5888,         # Tier 2 (100%-active Shared SwiGLU Expert): 45.22M/anchor (0.362B active)
    'ENG_EXPERTS': 32,               # Tier 3 (Routed SwiGLU Memory Experts): E=32 per anchor (256 experts total)
    'ENG_INITIAL_ACTIVE_EXPERTS': 8, # Coarse warmup macro-experts (8 -> upcycled to 32 at Stage 3 R24)
    'ENG_TOP_K': 7,                  # Top-7 of 32 active per token (21.9% coverage -> 1.01B Active Engram)
    'ENG_INITIAL_TOP_K': 4,          # Top-4 of 8 active during coarse warmup
    'ENG_EXPERT_RANK': 3072,         # 352.32M/anchor across 32 experts (2.819B total / 0.617B active)
    'ENG_ROUTE_DIM': 64,             # X-MoE hypersphere cosine router dimension
    'ENG_Z_LOSS_COEFF': 1.0e-4,      # ST-MoE router Z-loss coefficient
    'OFFLOAD_TIER1B_CPU': True,      # Pin Tier 1B in 130 GB Host DRAM with coalesced sparse row Adam

    # Full-Parameter + Tiered MoE-Engram Optimization
    'FULL_FINETUNE': True,
    'USE_8BIT_OPTIM': True,
    'LR_ADAPTER': 2.0e-4,            # MoE-Engram Tier 1A/1B/2/3 & Pass Adapters
    'LR_FULL': 2.0e-6,               # 36-Layer Spark Backbone (protective 100x lower LR)
    'LR_MHC': 5.0e-3,                # mHC routing logits & zero-initialized gates
    'WEIGHT_DECAY': 0.01,
    'GRAD_CLIP': 1.0,
    'SEQ_LEN': 2048,
    'MICRO_BATCH': 2,
    'GRAD_ACCUM': 8,
    'CE_CHUNK_TOKENS': 512,

    # 4-Stage Progressive Depth & Coarse-to-Fine Upcycling Schedule
    'STAGES': [
        {
            'name': 'warmup',
            'start': 4, 'end': 28, 'passes': 1,
            'enable_p3': False, 'p3_start': 12, 'p3_end': 24,
            'mhc': False, 'engram': True,
            'freeze_backbone': True,
            'upcycle_32': False,
            'steps': 60,
            'desc': '36L | Freeze backbone; warmup Tier-2 Shared + 8 Coarse Macro-Experts + Tier-1A/1B',
        },
        {
            'name': 'R12',
            'start': 8, 'end': 20, 'passes': 2,
            'enable_p3': False, 'p3_start': 12, 'p3_end': 24,
            'mhc': True, 'engram': True,
            'freeze_backbone': False,
            'upcycle_32': False,
            'steps': 120,
            'desc': '48L | Unfreeze full backbone (lr=2e-6) + 2-pass recurrence on [8..20) with 8 Coarse Experts',
        },
        {
            'name': 'R24',
            'start': 4, 'end': 28, 'passes': 2,
            'enable_p3': False, 'p3_start': 12, 'p3_end': 24,
            'mhc': True, 'engram': True,
            'freeze_backbone': False,
            'upcycle_32': True,
            'steps': 160,
            'desc': '60L | Upcycle 8 -> 32 Fine-Grained Experts (Top-K=7, 1.0B Active Engram) across [4..28)',
        },
        {
            'name': 'R72',
            'start': 4, 'end': 28, 'passes': 2,
            'enable_p3': True, 'p3_start': 12, 'p3_end': 24,
            'mhc': True, 'engram': True,
            'freeze_backbone': False,
            'upcycle_32': False,
            'steps': 160,
            'desc': '72L | Activate Pass-3 deep inner-core recurrence [12..24) (72 virtual layers total)',
        },
    ],

    # Data & Filtering Controls
    'REQUIRE_PLAN_AND_EXECUTION': True,
    'ENABLE_ANTI_SLOP_FILTER': True,
    'MASK_TOOL_RESPONSE_LABELS': True,
    'OPENCODE_BASE_URL': 'https://opencode.ai/zen/go/v1',
    'OPENCODE_MODEL': 'glm-5',
    'OPENCODE_ENRICH_SAMPLES': 24,

    # 5-Pillar, 46-Source High-Signal Dataset Curriculum
    'CURATED_SOURCES': [
        # --- PILLAR 1: Cognitive Planning, Architecture & Deep Reasoning (32%) ---
        {'id': 'LeafSpark/Fable-4B-Thinking-13K', 'pillar': 'planning', 'cap': 600},
        {'id': 'TeichAI/claude-4.5-opus-high-reasoning-250x', 'pillar': 'planning', 'cap': 250},
        {'id': 'TeichAI/claude-opus-4.6-800x', 'pillar': 'planning', 'cap': 500},
        {'id': 'crownelius/Opus-4.6-Reasoning-3300x', 'pillar': 'planning', 'cap': 700},
        {'id': 'LocoreMind/LocoTrainer-4B-Reasoning-Traces', 'pillar': 'planning', 'cap': 500},
        {'id': 'LocoreMind/Claude-Opus-4-6-Reasoning-Overlay', 'pillar': 'planning', 'cap': 500},
        {'id': 'mercor/apex-agents', 'pillar': 'planning', 'cap': 400},
        {'id': 'voidful/reasoning_plan_dataset_v0.2', 'pillar': 'planning', 'cap': 400},
        {'id': 'Subh775/System-Design', 'pillar': 'planning', 'cap': 300},
        {'id': 'Nothingger/planning-reasoning-dataset', 'pillar': 'planning', 'cap': 400},
        {'id': 'Quazim0t0/MegaPlanning-Dataset', 'pillar': 'planning', 'cap': 400},
        {'id': 'Tesslate/UIGEN-T1.5-Dataset', 'pillar': 'planning', 'cap': 400},
        {'id': 'TeichAI/gpt-5.4-high-reasoning-250x', 'pillar': 'planning', 'cap': 250},
        {'id': 'nohurry/Opus-4.6-Reasoning-3000x-filtered', 'pillar': 'planning', 'cap': 600},
        {'id': 'Locke6666/Astra-5-6-Reasoning-SFT-10K', 'pillar': 'planning', 'cap': 600},
        {'id': 'Locke6666/GPT-Astra-5.6-3500x', 'pillar': 'planning', 'cap': 500},

        # --- PILLAR 2: Verifiable STEM, Math & Code Execution (23%) ---
        {'id': 'open-r1/OpenR1-Math-220k', 'pillar': 'stem_code', 'cap': 600},
        {'id': 'open-thoughts/OpenThoughts-114k', 'pillar': 'stem_code', 'cap': 600},
        {'id': 'nvidia/Llama-Nemotron-Post-Training-Dataset', 'pillar': 'stem_code', 'cap': 500},
        {'id': 'bespokelabs/Bespoke-Stratos-17k', 'pillar': 'stem_code', 'cap': 400},
        {'id': 'ServiceNow-AI/R1-Distill-SFT', 'pillar': 'stem_code', 'cap': 400},
        {'id': 'cognitivecomputations/dolphin-r1', 'pillar': 'stem_code', 'cap': 400},
        {'id': 'FreedomIntelligence/medical-o1-reasoning-SFT', 'pillar': 'stem_code', 'cap': 350},
        {'id': 'GeneralReasoning/GeneralThought-430K', 'pillar': 'stem_code', 'cap': 400},
        {'id': 'OpenCoder-LLM/opc-sft-stage2', 'pillar': 'stem_code', 'cap': 400},
        {'id': 'Qwen/CodeElo', 'pillar': 'stem_code', 'cap': 250},
        {'id': 'HuggingFaceH4/MATH-500', 'pillar': 'stem_code', 'cap': 250},
        {'id': 'AI-MO/NuminaMath-CoT', 'pillar': 'stem_code', 'cap': 350},
        {'id': 'Magpie-Align/Magpie-Reasoning-V2-250K-CoT-Deepseek-R1-Llama-70B', 'pillar': 'stem_code', 'cap': 400},

        # --- PILLAR 3: Agentic Tool Calling, MCP & Interleaved Thinking (20%) ---
        {'id': 'Agent-Ark/Toucan-1.5M', 'pillar': 'agentic_tool', 'cap': 600},
        {'id': 'Team-ACE/ToolACE', 'pillar': 'agentic_tool', 'cap': 600},
        {'id': 'Salesforce/xlam-function-calling-60k', 'pillar': 'agentic_tool', 'cap': 500},
        {'id': 'Salesforce/APIGen-MT-5k', 'pillar': 'agentic_tool', 'cap': 400},
        {'id': 'hxssgaa/xlam-interleave-thinking-40k', 'pillar': 'agentic_tool', 'cap': 600},
        {'id': 'pyromind/agentic-tool-call-dataset-12k', 'pillar': 'agentic_tool', 'cap': 450},
        {'id': 'NousResearch/hermes-function-calling-v1', 'pillar': 'agentic_tool', 'cap': 400},
        {'id': 'SWE-Gym/SWE-Gym', 'pillar': 'agentic_tool', 'cap': 300},

        # --- PILLAR 4: Natural Human Language, World Knowledge & Anti-Slop Prose (15%) ---
        {'id': 'HuggingFaceH4/no_robots', 'pillar': 'human_prose', 'cap': 600},
        {'id': 'OpenAssistant/oasst2', 'pillar': 'human_prose', 'cap': 500},
        {'id': 'SAA-Lab/LitBench', 'pillar': 'human_prose', 'cap': 400},
        {'id': 'allenai/tulu-3-sft-mixture', 'pillar': 'human_prose', 'cap': 500},
        {'id': 'nvidia/HelpSteer2', 'pillar': 'human_prose', 'cap': 400},
        {'id': 'HuggingFaceTB/Cosmopedia-v2', 'pillar': 'human_prose', 'cap': 400},

        # --- PILLAR 5: Multi-Turn Conversational Coherence (10%) ---
        {'id': 'HuggingFaceH4/ultrachat_200k', 'pillar': 'chat', 'cap': 400},
        {'id': 'Magpie-Align/Magpie-Pro-300K-Filtered', 'pillar': 'chat', 'cap': 400},
        {'id': 'lmsys/lmsys-chat-1m', 'pillar': 'chat', 'cap': 300},
    ],
}
print(f"Configured {len(CFG['CURATED_SOURCES'])} dataset sources across 5 pillars for {CFG['HF_HUB_REPO_ID']}.")


In [ ]:
# Cell 4: Hardware Detection, Colab Secrets (HF_TOKEN / OPENCODE_API_KEY), and VRAM Auto-Scaling Guard
import os
import psutil
import torch
from huggingface_hub import login


def _get_secret(names: list[str]) -> str:
    for n in names:
        val = os.environ.get(n, '').strip()
        if val:
            return val
    try:
        from google.colab import userdata
        for n in names:
            try:
                val = userdata.get(n)
                if val and str(val).strip():
                    os.environ[n] = str(val).strip()
                    return str(val).strip()
            except Exception:
                pass
    except Exception:
        pass
    return ''


HF_TOKEN = _get_secret(['HF_TOKEN', 'HUGGINGFACE_TOKEN'])
OPENCODE_API_KEY = _get_secret(['OPENCODE_API_KEY', 'OPENCODE_API'])

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('Authenticated with Hugging Face Hub.')
    except Exception as exc:
        print(f'HF login warning: {exc}')
else:
    print('WARNING: HF_TOKEN not found in Colab Secrets; Hub push will be skipped unless set.')

if OPENCODE_API_KEY:
    print('OpenCode Go API key detected for teacher verification.')
else:
    print('INFO: OPENCODE_API_KEY not set; teacher verification pass will gracefully skip remote API calls.')

sys_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    gpu_name = 'CPU'
    vram_gb = 0.0

print(f'Detected Hardware: GPU = {gpu_name} ({vram_gb:.1f} GB VRAM) | System RAM = {sys_ram_gb:.1f} GB')

# Auto-scale micro-batch & sequence length to fit available VRAM without OOM
if vram_gb >= 75.0:
    CFG['MICRO_BATCH'] = 2
    CFG['GRAD_ACCUM'] = 8
    CFG['SEQ_LEN'] = 2048
    CFG['OFFLOAD_CHECKPOINT_CPU'] = False
elif vram_gb >= 38.0:
    CFG['MICRO_BATCH'] = 1
    CFG['GRAD_ACCUM'] = 16
    CFG['SEQ_LEN'] = 2048
    CFG['OFFLOAD_CHECKPOINT_CPU'] = False
else:
    CFG['MICRO_BATCH'] = 1
    CFG['GRAD_ACCUM'] = 16
    CFG['SEQ_LEN'] = 1024
    CFG['OFFLOAD_CHECKPOINT_CPU'] = True

print(
    f"Auto-Scaled Runtime Settings: MICRO_BATCH={CFG['MICRO_BATCH']}, "
    f"GRAD_ACCUM={CFG['GRAD_ACCUM']}, SEQ_LEN={CFG['SEQ_LEN']}, "
    f"OFFLOAD_TIER1B_CPU={CFG['OFFLOAD_TIER1B_CPU']}"
)


In [ ]:
# Cell 5: Write Self-Contained `sparkx_loop_v03_core.py` and `load_inference_v03.py`
from pathlib import Path

CORE_SOURCE = '"""SparkX-Loop v0.3 (`Spark-X2.6-4B-En4B-A1B`):\n8.2B Total / 5.2B Active Parameters | 72 Effective Layers (3-Pass Telescoping Recurrence) |\n3-Tier Zipfian MoE-Engram (4.0B Total / 1.0B Active across 8 Macro-Block Anchors) with\nHost DRAM / NVMe mmap Tier-1B Offloading, Coalesced Sparse Row Adam, 8->32 Coarse-to-Fine\nExpert Upcycling, Anti-Collapse Hypersphere Cosine + Sigmoid + Z-Loss Routing, and\nRuntime-Only Zero-Collapse O(n log n) Log-Linear NoPE KV Cache.\n"""\nfrom __future__ import annotations\nfrom contextlib import nullcontext\nimport copy\nimport dataclasses\nimport importlib\nimport math\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\ndef _probe_sdpa_gqa() -> bool:\n    if not hasattr(F, \'scaled_dot_product_attention\'):\n        return False\n    try:\n        q = torch.zeros((1, 2, 1, 4), dtype=torch.float32)\n        k = torch.zeros((1, 1, 1, 4), dtype=torch.float32)\n        F.scaled_dot_product_attention(q, k, k, enable_gqa=True)\n        return True\n    except Exception:\n        return False\n\n\n_SDPA_SUPPORTS_GQA = _probe_sdpa_gqa()\n\n\n@dataclasses.dataclass\nclass LoopV03Settings:\n    start: int = 4\n    end: int = 28                  # Pass-1 & Pass-2 recurrent core [4..28) (24 layers)\n    passes: int = 2                # Base core passes (1 or 2)\n    p3_start: int = 12             # Pass-3 deep inner core start [12..24) (12 layers -> 72 total virtual layers)\n    p3_end: int = 24               # Pass-3 deep inner core end\n    enable_p3: bool = False        # When True with passes=2: 4 + 24 + 24 + 12 + 8 = 72 effective layers\n    max_passes: int = 4\n    mhc_streams: int = 2\n    # 3-Tier Zipfian 4.0B-A1B MoE-Engram settings (per anchor across 8 anchors: Layers 3,7,11,15,19,23,27,31)\n    eng_hot_buckets: int = 16384   # Tier 1A (GPU HBM hot cache): 16,384 x 512 x 3 = 25.17M/anchor (0.20B total)\n    eng_cold_buckets: int = 49152  # Tier 1B (Host DRAM / mmap cold bank): 49,152 x 512 x 3 = 75.50M/anchor (0.60B total)\n    eng_dim: int = 512             # Lexical & routing embedding width\n    eng_shared_rank: int = 5888    # Tier 2 (100%-active Shared SwiGLU Expert): 2560->5888->2560 = 45.22M/anchor (0.36B active)\n    eng_experts: int = 32          # Tier 3 (Routed SwiGLU Memory Experts): E = 32 experts per anchor (256 total)\n    eng_initial_active_experts: int = 8  # Stage 1/2 coarse macro-experts (8 -> upcycled to 32 at Stage 3)\n    eng_top_k: int = 7             # Top-7 of 32 active per token (21.9% coverage) -> 0.62B active across 8 anchors\n    eng_initial_top_k: int = 4     # Top-4 of 8 active during coarse warmup\n    eng_expert_rank: int = 3072    # SwiGLU rank per routed expert: (512*3072*2 + 3072*2560)*32 = 352.32M/anchor (2.82B total)\n    eng_route_dim: int = 64        # X-MoE hypersphere cosine router subspace dimension\n    eng_z_loss_coeff: float = 1e-4 # ST-MoE router Z-loss coefficient\n    eng_offload_tier1b_cpu: bool = True  # Pin Tier 1B in 130 GB Host DRAM with async prefetch + coalesced sparse Adam\n    backbone_engram_stride: int = 4      # Anchors at Layers 3, 7, 11, 15, 19, 23, 27, 31 (8 anchors total)\n    num_engram_anchors: int = 8\n    checkpoint_layers: bool = True\n    offload_checkpoint_to_cpu: bool = False\n    use_mhc: bool = True\n    use_engram: bool = True\n    dynamic_mhc: bool = True\n    sublayer_mhc: bool = True\n    pass_adapter_rank: int = 16\n    ce_chunk_tokens: int = 512\n\n\nclass _CacheLayersView:\n    def __init__(self, cache, physical: int, virtual: int):\n        self.cache = cache\n        self.physical = physical\n        self.virtual = virtual\n\n    def __getitem__(self, index: int):\n        if index != self.physical:\n            raise IndexError(\'A physical layer may only access its own virtual cache slot\')\n        return self.cache.layers[self.virtual]\n\n\nclass _VisitCache:\n    """Redirects physical layer_idx KV cache updates to unique per-visit virtual_idx slots (up to 72 visits),\n    and enforces BF16/FP16 KV cache storage so RoPE float32 intermediates never bloat KV memory.\n    """\n    def __init__(self, cache, physical_idx: int, virtual_idx: int, kv_dtype: Optional[torch.dtype] = None):\n        self.cache = cache\n        self.physical_idx = physical_idx\n        self.virtual_idx = virtual_idx\n        self.kv_dtype = kv_dtype\n        if hasattr(cache, \'layers\'):\n            self.layers = _CacheLayersView(cache, physical_idx, virtual_idx)\n\n    def update(self, key_states, value_states, layer_idx: int, cache_kwargs=None):\n        target = self.virtual_idx if layer_idx == self.physical_idx else layer_idx\n        if self.kv_dtype is not None:\n            if isinstance(key_states, torch.Tensor) and key_states.dtype != self.kv_dtype:\n                key_states = key_states.to(self.kv_dtype)\n            if isinstance(value_states, torch.Tensor) and value_states.dtype != self.kv_dtype:\n                value_states = value_states.to(self.kv_dtype)\n        elif isinstance(key_states, torch.Tensor) and isinstance(value_states, torch.Tensor):\n            if key_states.dtype == torch.float32 and value_states.dtype in (torch.bfloat16, torch.float16):\n                key_states = key_states.to(value_states.dtype)\n        return self.cache.update(key_states, value_states, target, cache_kwargs)\n\n    def __getattr__(self, name):\n        return getattr(self.cache, name)\n\n\ndef patch_spark_attention_for_long_context(base_model: nn.Module) -> int:\n    """Replace Spark2_5Attention\'s O(T^2) eager_attention_forward with fused BF16 PyTorch SDPA / GQA."""\n    if not hasattr(base_model, \'model\') or not hasattr(base_model.model, \'layers\'):\n        return 0\n    layers = base_model.model.layers\n    if not layers or not hasattr(layers[0], \'self_attn\'):\n        return 0\n    default_sw = int(getattr(getattr(base_model, \'config\', None), \'sliding_window\', 512) or 512)\n    for lyr in layers:\n        is_sl = (\n            getattr(lyr, \'layer_type\', \'\') == \'sliding_attention\'\n            or bool(getattr(lyr.self_attn, \'sliding_window\', None))\n            or bool(getattr(lyr.self_attn, \'is_sliding\', False))\n        )\n        lyr.self_attn._sparkx_sliding_window = int(getattr(lyr.self_attn, \'sliding_window\', None) or default_sw) if is_sl else None\n\n    attn_mod = importlib.import_module(type(layers[0].self_attn).__module__)\n    if not hasattr(attn_mod, \'eager_attention_forward\'):\n        return 0\n    if getattr(attn_mod, \'_sparkx_sdpa_patched\', False) and not getattr(attn_mod, \'_sparkx_loglinear_patched\', False):\n        return len(layers)\n\n    orig_repeat_kv = getattr(attn_mod, \'repeat_kv\', None)\n\n    def fused_sdpa_attention_forward(\n        module: nn.Module,\n        query: torch.Tensor,\n        key: torch.Tensor,\n        value: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        scaling: Optional[float] = None,\n        dropout: float = 0.0,\n        **kwargs,\n    ):\n        if scaling is None:\n            scaling = 1.0 / math.sqrt(query.shape[-1])\n        qkv_proj = getattr(module, \'q_k_v_proj\', None)\n        target_dtype = getattr(qkv_proj, \'weight\', value).dtype if qkv_proj is not None else value.dtype\n        if query.dtype != target_dtype:\n            query = query.to(target_dtype)\n        if key.dtype != target_dtype:\n            key = key.to(target_dtype)\n        if value.dtype != target_dtype:\n            value = value.to(target_dtype)\n\n        sw = getattr(module, \'_sparkx_sliding_window\', None) or getattr(module, \'sliding_window\', None)\n        if sw is not None and attention_mask is None and query.shape[-2] == 1 and key.shape[-2] > int(sw):\n            key = key[:, :, -int(sw):, :]\n            value = value[:, :, -int(sw):, :]\n\n        attn_mask = None\n        if attention_mask is not None:\n            attn_mask = attention_mask[:, :, :, : key.shape[-2]]\n            if attn_mask.dtype != torch.bool and attn_mask.dtype != query.dtype:\n                attn_mask = attn_mask.to(query.dtype)\n        use_gqa = _SDPA_SUPPORTS_GQA and (module.num_key_value_groups > 1)\n        if not use_gqa and orig_repeat_kv is not None and module.num_key_value_groups > 1:\n            key = orig_repeat_kv(key, module.num_key_value_groups)\n            value = orig_repeat_kv(value, module.num_key_value_groups)\n        sdpa_kwargs = {\'scale\': scaling}\n        if use_gqa:\n            sdpa_kwargs[\'enable_gqa\'] = True\n        out = F.scaled_dot_product_attention(\n            query, key, value,\n            attn_mask=attn_mask,\n            dropout_p=dropout if module.training else 0.0,\n            is_causal=False,\n            **sdpa_kwargs,\n        )\n        return out, None\n\n    attn_mod.eager_attention_forward = fused_sdpa_attention_forward\n    attn_mod._sparkx_sdpa_patched = True\n    attn_mod._sparkx_loglinear_patched = False\n    return len(layers)\n\n\ndef _compress_nope_kv_loglinear(\n    query: torch.Tensor,\n    key: torch.Tensor,\n    value: torch.Tensor,\n    sink_tokens: int = 32,\n    local_window: int = 1024,\n    base_block: int = 32,\n    hip_top_blocks: int = 8,\n) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n    """Log-Mass-Conserved Dyadic Fenwick Tree + HiP Top-M Exact Block selection for NoPE KV caches."""\n    bsz, n_kv_heads, seq_k, head_dim = key.shape\n    device = key.device\n    dtype = query.dtype\n\n    mid_start = min(sink_tokens, seq_k)\n    mid_end = max(mid_start, seq_k - local_window)\n    mid_len = mid_end - mid_start\n    n_blocks = mid_len // base_block\n\n    if n_blocks <= hip_top_blocks + 2:\n        zeros = torch.zeros((bsz, 1, 1, seq_k), dtype=dtype, device=device)\n        return key, value, zeros\n\n    usable_mid_end = mid_start + n_blocks * base_block\n    k_sink = key[:, :, :mid_start, :]\n    v_sink = value[:, :, :mid_start, :]\n    k_local = key[:, :, usable_mid_end:, :]\n    v_local = value[:, :, usable_mid_end:, :]\n\n    k_mid = key[:, :, mid_start:usable_mid_end, :].view(bsz, n_kv_heads, n_blocks, base_block, head_dim)\n    v_mid = value[:, :, mid_start:usable_mid_end, :].view(bsz, n_kv_heads, n_blocks, base_block, head_dim)\n\n    k_mid_f = k_mid.float()\n    v_mid_f = v_mid.float()\n    block_mean_k = k_mid_f.mean(dim=3)\n    block_mean_norm = k_mid_f.norm(dim=-1, keepdim=True).mean(dim=3)\n    centroid_k = block_mean_k * (block_mean_norm / (block_mean_k.norm(dim=-1, keepdim=True) + 1e-6))\n    centroid_v = v_mid_f.mean(dim=3)\n\n    q_ref = query[:, :, -min(4, query.shape[2]):, :].float().mean(dim=2)\n    if q_ref.shape[1] != n_kv_heads:\n        groups = q_ref.shape[1] // n_kv_heads\n        q_ref_kv = q_ref.view(bsz, n_kv_heads, groups, head_dim).mean(dim=2)\n    else:\n        q_ref_kv = q_ref\n    block_scores = torch.einsum(\'bhd,bhnd->bn\', q_ref_kv, centroid_k) / math.sqrt(head_dim)\n    top_m = min(hip_top_blocks, n_blocks)\n    _, top_idx = torch.topk(block_scores, k=top_m, dim=-1, sorted=True)\n\n    top_set = set(top_idx[0].tolist())\n    exact_k_list = [k_sink]\n    exact_v_list = [v_sink]\n    for b_idx in sorted(top_set):\n        exact_k_list.append(k_mid[:, :, b_idx, :, :])\n        exact_v_list.append(v_mid[:, :, b_idx, :, :])\n\n    sum_k_list, sum_v_list, sum_log_mass = [], [], []\n    i = 0\n    while i < n_blocks:\n        if i in top_set:\n            i += 1\n            continue\n        span = 1\n        while (i + span * 2 <= n_blocks) and all((i + j) not in top_set for j in range(span * 2)):\n            if span >= 16:\n                break\n            span *= 2\n        if span == 1:\n            sum_k_list.append(centroid_k[:, :, i : i + 1, :].to(key.dtype))\n            sum_v_list.append(centroid_v[:, :, i : i + 1, :].to(value.dtype))\n        else:\n            sub_k = centroid_k[:, :, i : i + span, :]\n            sub_norm = sub_k.norm(dim=-1, keepdim=True).mean(dim=2, keepdim=True)\n            sub_mean = sub_k.mean(dim=2, keepdim=True)\n            merged_k = sub_mean * (sub_norm / (sub_mean.norm(dim=-1, keepdim=True) + 1e-6))\n            merged_v = centroid_v[:, :, i : i + span, :].mean(dim=2, keepdim=True)\n            sum_k_list.append(merged_k.to(key.dtype))\n            sum_v_list.append(merged_v.to(value.dtype))\n        sum_log_mass.append(math.log(float(span * base_block)))\n        i += span\n\n    exact_k_list.append(k_local)\n    exact_v_list.append(v_local)\n    k_exact_all = torch.cat(exact_k_list, dim=2)\n    v_exact_all = torch.cat(exact_v_list, dim=2)\n    n_exact = k_exact_all.shape[2]\n\n    if sum_k_list:\n        k_sum_all = torch.cat(sum_k_list, dim=2)\n        v_sum_all = torch.cat(sum_v_list, dim=2)\n        k_out = torch.cat([k_exact_all, k_sum_all], dim=2)\n        v_out = torch.cat([v_exact_all, v_sum_all], dim=2)\n        bias = torch.zeros((bsz, 1, 1, n_exact + len(sum_log_mass)), dtype=dtype, device=device)\n        bias[:, :, :, n_exact:] = torch.tensor(sum_log_mass, dtype=dtype, device=device).view(1, 1, 1, -1)\n        return k_out, v_out, bias\n\n    bias = torch.zeros((bsz, 1, 1, n_exact), dtype=dtype, device=device)\n    return k_exact_all, v_exact_all, bias\n\n\ndef patch_spark_attention_for_loglinear_kv(\n    base_model: nn.Module,\n    exact_threshold: int = 2048,\n    sink_tokens: int = 32,\n    local_window: int = 1024,\n    base_block: int = 32,\n    hip_top_blocks: int = 8,\n) -> int:\n    """Runtime-only O(n^2) -> O(n log n) KV cache optimization for Spark-X2.6-4B\'s NoPE layers."""\n    if not hasattr(base_model, \'model\') or not hasattr(base_model.model, \'layers\'):\n        return 0\n    layers = base_model.model.layers\n    if not layers or not hasattr(layers[0], \'self_attn\'):\n        return 0\n    default_sw = int(getattr(getattr(base_model, \'config\', None), \'sliding_window\', 512) or 512)\n    for lyr in layers:\n        is_sl = (\n            getattr(lyr, \'layer_type\', \'\') == \'sliding_attention\'\n            or bool(getattr(lyr.self_attn, \'sliding_window\', None))\n            or bool(getattr(lyr.self_attn, \'is_sliding\', False))\n        )\n        lyr.self_attn._sparkx_sliding_window = int(getattr(lyr.self_attn, \'sliding_window\', None) or default_sw) if is_sl else None\n\n    attn_mod = importlib.import_module(type(layers[0].self_attn).__module__)\n    orig_repeat_kv = getattr(attn_mod, \'repeat_kv\', None)\n\n    def loglinear_sdpa_attention_forward(\n        module: nn.Module,\n        query: torch.Tensor,\n        key: torch.Tensor,\n        value: torch.Tensor,\n        attention_mask: Optional[torch.Tensor] = None,\n        scaling: Optional[float] = None,\n        dropout: float = 0.0,\n        **kwargs,\n    ):\n        if scaling is None:\n            scaling = 1.0 / math.sqrt(query.shape[-1])\n        qkv_proj = getattr(module, \'q_k_v_proj\', None)\n        target_dtype = getattr(qkv_proj, \'weight\', value).dtype if qkv_proj is not None else value.dtype\n        if query.dtype != target_dtype:\n            query = query.to(target_dtype)\n        if key.dtype != target_dtype:\n            key = key.to(target_dtype)\n        if value.dtype != target_dtype:\n            value = value.to(target_dtype)\n\n        sw = getattr(module, \'_sparkx_sliding_window\', None) or getattr(module, \'sliding_window\', None)\n        is_nope_full = (sw is None) and (not getattr(module, \'is_sliding\', False))\n        if sw is not None and attention_mask is None and query.shape[-2] == 1 and key.shape[-2] > int(sw):\n            key = key[:, :, -int(sw):, :]\n            value = value[:, :, -int(sw):, :]\n\n        seq_q, seq_k = query.shape[-2], key.shape[-2]\n\n        if (\n            not module.training\n            and is_nope_full\n            and seq_k > exact_threshold\n            and seq_q <= 64\n            and query.shape[0] == 1\n        ):\n            k_comp, v_comp, log_mass_bias = _compress_nope_kv_loglinear(\n                query, key, value,\n                sink_tokens=sink_tokens,\n                local_window=local_window,\n                base_block=base_block,\n                hip_top_blocks=hip_top_blocks,\n            )\n            use_gqa = _SDPA_SUPPORTS_GQA and (module.num_key_value_groups > 1)\n            if not use_gqa and orig_repeat_kv is not None and module.num_key_value_groups > 1:\n                k_comp = orig_repeat_kv(k_comp, module.num_key_value_groups)\n                v_comp = orig_repeat_kv(v_comp, module.num_key_value_groups)\n            sdpa_kwargs = {\'scale\': scaling}\n            if use_gqa:\n                sdpa_kwargs[\'enable_gqa\'] = True\n            out = F.scaled_dot_product_attention(\n                query, k_comp, v_comp,\n                attn_mask=log_mass_bias.expand(query.shape[0], 1, seq_q, k_comp.shape[-2]),\n                dropout_p=0.0,\n                is_causal=False,\n                **sdpa_kwargs,\n            )\n            return out, None\n\n        attn_mask = None\n        if attention_mask is not None:\n            attn_mask = attention_mask[:, :, :, : key.shape[-2]]\n            if attn_mask.dtype != torch.bool and attn_mask.dtype != query.dtype:\n                attn_mask = attn_mask.to(query.dtype)\n        use_gqa = _SDPA_SUPPORTS_GQA and (module.num_key_value_groups > 1)\n        if not use_gqa and orig_repeat_kv is not None and module.num_key_value_groups > 1:\n            key = orig_repeat_kv(key, module.num_key_value_groups)\n            value = orig_repeat_kv(value, module.num_key_value_groups)\n        sdpa_kwargs = {\'scale\': scaling}\n        if use_gqa:\n            sdpa_kwargs[\'enable_gqa\'] = True\n        out = F.scaled_dot_product_attention(\n            query, key, value,\n            attn_mask=attn_mask,\n            dropout_p=dropout if module.training else 0.0,\n            is_causal=False,\n            **sdpa_kwargs,\n        )\n        return out, None\n\n    attn_mod.eager_attention_forward = loglinear_sdpa_attention_forward\n    attn_mod._sparkx_sdpa_patched = True\n    attn_mod._sparkx_loglinear_patched = True\n    return len(layers)\n\n\nclass ZipfianTieredMoEEngram(nn.Module):\n    """3-Tier Zipfian MoE-Engram (`En4B-A1B`: 500M Total / 125M Active per Anchor x 8 Anchors = 4.0B Total / 1.0B Active).\n\n    - Tier 1A (GPU HBM Hot Lexical Cache): `hot_buckets` x `eng_dim` x 3 scales (2-gram, 3-gram, skip-2-gram), zero-init.\n    - Tier 1B (Host DRAM / NVMe mmap Cold N-Gram Bank): `cold_buckets` x `eng_dim` x 3 scales, zero-init, with\n      deterministic ahead-of-time prefetch and coalesced sparse row Adam updates (`step_tier1b_sparse_adam`).\n    - Tier 2 (100%-Active Hot Shared SwiGLU Expert): `shared_gate_up` [width -> 2*shared_rank] + `shared_down` [shared_rank -> width].\n    - Tier 3 (E=32 Routed SwiGLU Memory Experts, Top-K=7):\n      - X-MoE L2-normalized Hypersphere Cosine Router + DeepSeek-V3 Sigmoid Affinity + Auxiliary-Loss-Free Bias\n      - ST-MoE Router Z-Loss (`last_router_z_loss`)\n      - Coarse-to-Fine Upcycling (`active_experts = 8 -> 32` via `upcycle_experts_8_to_32()`)\n    - Context-Aware Gate `2 * sigmoid(W_ctx h_t)` + Zero-Init `project` (`max_abs_diff == 0.0` at Step 0).\n    """\n    def __init__(\n        self,\n        hot_buckets: int = 16384,\n        cold_buckets: int = 49152,\n        eng_dim: int = 512,\n        width: int = 2560,\n        shared_rank: int = 5888,\n        num_experts: int = 32,\n        initial_active_experts: int = 8,\n        top_k: int = 7,\n        initial_top_k: int = 4,\n        expert_rank: int = 3072,\n        route_dim: int = 64,\n        max_passes: int = 4,\n        offload_tier1b_cpu: bool = True,\n        param_dtype: torch.dtype = torch.bfloat16,\n    ):\n        super().__init__()\n        self.hot_buckets = int(hot_buckets)\n        self.cold_buckets = int(cold_buckets)\n        self.total_buckets = self.hot_buckets + self.cold_buckets\n        self.eng_dim = int(eng_dim)\n        self.width = int(width)\n        self.shared_rank = int(shared_rank)\n        self.num_experts = int(num_experts)\n        self.active_experts = min(int(initial_active_experts), self.num_experts)\n        self.target_top_k = min(int(top_k), self.num_experts)\n        self.active_top_k = min(int(initial_top_k), self.active_experts)\n        self.expert_rank = int(expert_rank)\n        self.route_dim = int(route_dim)\n        self.max_passes = int(max_passes)\n        self.offload_tier1b_cpu = bool(offload_tier1b_cpu)\n\n        # Tier 1A: GPU HBM Hot Lexical Cache (3 scales * hot_buckets, zero-init for cold-bucket safety)\n        self.tier1a_hot = nn.Embedding(self.hot_buckets * 3, self.eng_dim, dtype=param_dtype)\n        nn.init.zeros_(self.tier1a_hot.weight)\n        self.scale_weights = nn.Parameter(torch.zeros(3, dtype=torch.float32))\n\n        # Tier 1B: Host DRAM / mmap Long-Tail N-Gram Bank (not registered as a dense GPU Parameter when offloaded)\n        if self.cold_buckets > 0 and self.offload_tier1b_cpu:\n            pin = torch.cuda.is_available()\n            self.tier1b_cold_weight = torch.zeros(\n                (self.cold_buckets * 3, self.eng_dim), dtype=param_dtype, device=\'cpu\', pin_memory=pin\n            )\n            self.tier1b_exp_avg = torch.zeros_like(self.tier1b_cold_weight, pin_memory=pin)\n            self.tier1b_exp_avg_sq = torch.zeros_like(self.tier1b_cold_weight, pin_memory=pin)\n            self.tier1b_step = 0\n            self._pending_tier1b_rows: Optional[torch.Tensor] = None\n            self._pending_tier1b_leaf: Optional[torch.Tensor] = None\n        elif self.cold_buckets > 0:\n            self.tier1b_cold = nn.Embedding(self.cold_buckets * 3, self.eng_dim, dtype=param_dtype)\n            nn.init.zeros_(self.tier1b_cold.weight)\n        else:\n            self.tier1b_cold_weight = None\n\n        self.norm_eps = 1e-6\n        self.norm_weight = nn.Parameter(torch.ones(self.width, dtype=torch.float32))\n\n        # Lexical projection to width for conditioning Tier-2 and Tier-3\n        self.lexical_up = nn.Linear(self.eng_dim, self.width, bias=False, dtype=param_dtype)\n        nn.init.normal_(self.lexical_up.weight, mean=0.0, std=0.02)\n\n        # Tier 2: 100%-Active Hot Shared SwiGLU Expert (2560 -> 5888 -> 2560 = 45.22M params/anchor)\n        self.shared_gate = nn.Linear(self.width, self.shared_rank, bias=False, dtype=param_dtype)\n        self.shared_up = nn.Linear(self.width, self.shared_rank, bias=False, dtype=param_dtype)\n        self.shared_down = nn.Linear(self.shared_rank, self.width, bias=False, dtype=param_dtype)\n        nn.init.kaiming_uniform_(self.shared_gate.weight, a=math.sqrt(5))\n        nn.init.kaiming_uniform_(self.shared_up.weight, a=math.sqrt(5))\n        nn.init.normal_(self.shared_down.weight, mean=0.0, std=0.01)\n\n        # Tier 3: Anti-Collapse X-MoE Hypersphere Cosine Router + DeepSeek-V3 Sigmoid Affinity\n        self.router_proj = nn.Linear(self.width, self.route_dim, bias=False, dtype=torch.float32)\n        self.expert_centroids = nn.Parameter(torch.empty(self.num_experts, self.route_dim, dtype=torch.float32))\n        nn.init.orthogonal_(self.router_proj.weight)\n        nn.init.orthogonal_(self.expert_centroids)\n        self.router_log_temp = nn.Parameter(torch.tensor(math.log(0.15), dtype=torch.float32))\n        self.pass_router_bias = nn.Parameter(torch.zeros(self.max_passes, self.num_experts, dtype=torch.float32))\n        self.register_buffer(\'expert_bias\', torch.zeros(self.num_experts, dtype=torch.float32))\n        self.register_buffer(\'expert_load_ema\', torch.full((self.num_experts,), 1.0 / self.num_experts, dtype=torch.float32))\n\n        # Tier 3: Routed SwiGLU Memory Experts (E=32, rank=1440 -> 353.89M params/anchor)\n        # Parameterized as SwiGLU over query_proj(h_t) + e_lex in R^eng_dim -> expert_rank -> width\n        # Wait: to reach exact 353.89M per anchor (or configurable rank), we use:\n        # w_gate: [E, eng_dim, expert_rank], w_up: [E, eng_dim, expert_rank], w_down: [E, expert_rank, width]\n        self.query_proj = nn.Linear(self.width, self.eng_dim, bias=False, dtype=param_dtype)\n        nn.init.kaiming_uniform_(self.query_proj.weight, a=math.sqrt(5))\n\n        self.expert_gate = nn.Parameter(torch.empty(self.num_experts, self.eng_dim, self.expert_rank, dtype=param_dtype))\n        self.expert_up = nn.Parameter(torch.empty(self.num_experts, self.eng_dim, self.expert_rank, dtype=param_dtype))\n        self.expert_down = nn.Parameter(torch.empty(self.num_experts, self.expert_rank, self.width, dtype=param_dtype))\n        self._init_orthogonal_experts()\n\n        # Context-aware sigmoid gate & zero-initialized residual projection\n        self.ctx_gate = nn.Linear(self.width, self.width, bias=True, dtype=param_dtype)\n        nn.init.zeros_(self.ctx_gate.weight)\n        nn.init.zeros_(self.ctx_gate.bias)\n\n        self.project = nn.Linear(self.width, self.width, bias=False, dtype=param_dtype)\n        nn.init.zeros_(self.project.weight)\n        self.gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))\n        self.last_router_z_loss: torch.Tensor = torch.tensor(0.0)\n\n    @torch.no_grad()\n    def _init_orthogonal_experts(self):\n        for e in range(self.num_experts):\n            nn.init.orthogonal_(self.expert_gate[e].float())\n            nn.init.orthogonal_(self.expert_up[e].float())\n            nn.init.orthogonal_(self.expert_down[e].float())\n        self.expert_gate.data.mul_(0.05)\n        self.expert_up.data.mul_(0.05)\n        self.expert_down.data.mul_(0.02)\n\n    @torch.no_grad()\n    def upcycle_experts_8_to_32(self, noise_std: float = 1e-3):\n        """Clone the 8 trained coarse macro-experts into 32 fine-grained experts with orthogonal perturbations."""\n        if self.active_experts >= self.num_experts:\n            self.active_top_k = self.target_top_k\n            return\n        coarse = self.active_experts\n        factor = self.num_experts // coarse\n        for m in range(coarse):\n            for sub in range(factor):\n                dst = m * factor + sub\n                if dst == m:\n                    continue\n                self.expert_gate.data[dst].copy_(self.expert_gate.data[m])\n                self.expert_up.data[dst].copy_(self.expert_up.data[m])\n                self.expert_down.data[dst].copy_(self.expert_down.data[m])\n                self.expert_centroids.data[dst].copy_(self.expert_centroids.data[m])\n                if sub > 0:\n                    self.expert_gate.data[dst].add_(torch.randn_like(self.expert_gate.data[dst]) * noise_std)\n                    self.expert_up.data[dst].add_(torch.randn_like(self.expert_up.data[dst]) * noise_std)\n                    self.expert_centroids.data[dst].add_(torch.randn_like(self.expert_centroids.data[dst]) * noise_std)\n        self.active_experts = self.num_experts\n        self.active_top_k = self.target_top_k\n        self.expert_load_ema.fill_(1.0 / self.num_experts)\n        self.expert_bias.zero_()\n\n    @staticmethod\n    def compute_ngram_indices(\n        input_ids: torch.LongTensor,\n        total_buckets: int,\n        prev_ids: Optional[torch.LongTensor] = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        if prev_ids is None:\n            p1 = F.pad(input_ids[:, :-1], (1, 0), value=0)\n            p2 = F.pad(input_ids[:, :-2], (2, 0), value=0)\n        else:\n            cat1 = torch.cat([prev_ids[:, -1:], input_ids[:, :-1]], dim=1)\n            if prev_ids.shape[1] >= 2:\n                cat2 = torch.cat([prev_ids[:, -2:], input_ids[:, :-2]], dim=1)\n            else:\n                pad2 = F.pad(prev_ids[:, -1:], (1, 0), value=0)\n                cat2 = torch.cat([pad2, input_ids[:, :-2]], dim=1)[:, : input_ids.shape[1]]\n            p1, p2 = cat1, cat2\n        bi = (p1 * 1315423911 + input_ids * 2654435761) % total_buckets\n        tri = (p2 * 97531 + p1 * 1315423911 + input_ids * 2654435761) % total_buckets\n        skip = (p2 * 65537 + input_ids * 2654435761) % total_buckets\n        return bi, tri, skip\n\n    def _lookup_tier1_lexical(\n        self,\n        ngram_indices: tuple[torch.Tensor, torch.Tensor, torch.Tensor],\n        device: torch.device,\n        param_dtype: torch.dtype,\n    ) -> torch.Tensor:\n        bi, tri, skip = ngram_indices\n        sw = F.softmax(self.scale_weights, dim=0).to(param_dtype)\n        hot_b = self.hot_buckets\n\n        # Hot Tier 1A lookup on GPU HBM\n        bi_hot = bi.clamp(max=hot_b - 1)\n        tri_hot = tri.clamp(max=hot_b - 1) + hot_b\n        skip_hot = skip.clamp(max=hot_b - 1) + 2 * hot_b\n\n        e_bi = self.tier1a_hot(bi_hot)\n        e_tri = self.tier1a_hot(tri_hot)\n        e_skip = self.tier1a_hot(skip_hot)\n\n        if self.cold_buckets <= 0:\n            return sw[0] * e_bi + sw[1] * e_tri + sw[2] * e_skip\n\n        is_cold_bi = (bi >= hot_b)\n        is_cold_tri = (tri >= hot_b)\n        is_cold_skip = (skip >= hot_b)\n\n        if not (is_cold_bi.any() or is_cold_tri.any() or is_cold_skip.any()):\n            return sw[0] * e_bi + sw[1] * e_tri + sw[2] * e_skip\n\n        cold_b = self.cold_buckets\n        c_bi = (bi - hot_b).clamp(min=0, max=cold_b - 1)\n        c_tri = (tri - hot_b).clamp(min=0, max=cold_b - 1) + cold_b\n        c_skip = (skip - hot_b).clamp(min=0, max=cold_b - 1) + 2 * cold_b\n\n        if self.offload_tier1b_cpu and self.tier1b_cold_weight is not None:\n            # Coalesced unique row prefetch from pinned Host DRAM -> GPU HBM\n            stacked = torch.stack([c_bi, c_tri, c_skip], dim=0)\n            uniq_gpu, inv = torch.unique(stacked, return_inverse=True)\n            uniq_cpu = uniq_gpu.cpu()\n            fetched_cpu = self.tier1b_cold_weight[uniq_cpu]\n            fetched_gpu = fetched_cpu.to(device=device, dtype=param_dtype, non_blocking=True)\n            if self.training:\n                fetched_gpu = fetched_gpu.detach().requires_grad_(True)\n                self._pending_tier1b_rows = uniq_cpu\n                self._pending_tier1b_leaf = fetched_gpu\n            gathered = fetched_gpu[inv]  # [3, B, T, eng_dim]\n            c_e_bi, c_e_tri, c_e_skip = gathered[0], gathered[1], gathered[2]\n        else:\n            c_e_bi = self.tier1b_cold(c_bi)\n            c_e_tri = self.tier1b_cold(c_tri)\n            c_e_skip = self.tier1b_cold(c_skip)\n\n        e_bi = torch.where(is_cold_bi.unsqueeze(-1), c_e_bi, e_bi)\n        e_tri = torch.where(is_cold_tri.unsqueeze(-1), c_e_tri, e_tri)\n        e_skip = torch.where(is_cold_skip.unsqueeze(-1), c_e_skip, e_skip)\n        return sw[0] * e_bi + sw[1] * e_tri + sw[2] * e_skip\n\n    @torch.no_grad()\n    def step_tier1b_sparse_adam(self, lr: float = 2e-4, beta1: float = 0.9, beta2: float = 0.95, eps: float = 1e-8):\n        """Coalesced sparse Adam update on pinned Host DRAM for Tier 1B (<0.3 ms/step)."""\n        if (\n            not self.offload_tier1b_cpu\n            or self.tier1b_cold_weight is None\n            or self._pending_tier1b_leaf is None\n            or self._pending_tier1b_leaf.grad is None\n            or self._pending_tier1b_rows is None\n        ):\n            self._pending_tier1b_leaf = None\n            self._pending_tier1b_rows = None\n            return\n        rows = self._pending_tier1b_rows\n        grad = self._pending_tier1b_leaf.grad.detach().to(device=\'cpu\', dtype=torch.float32)\n        self.tier1b_step += 1\n        m = self.tier1b_exp_avg[rows].float()\n        v = self.tier1b_exp_avg_sq[rows].float()\n        m.mul_(beta1).add_(grad, alpha=1.0 - beta1)\n        v.mul_(beta2).addcmul_(grad, grad, value=1.0 - beta2)\n        bc1 = 1.0 - (beta1 ** self.tier1b_step)\n        bc2 = 1.0 - (beta2 ** self.tier1b_step)\n        step_size = lr / bc1\n        denom = (v.sqrt() / math.sqrt(bc2)).add_(eps)\n        w = self.tier1b_cold_weight[rows].float()\n        w.addcdiv_(m, denom, value=-step_size)\n        self.tier1b_exp_avg[rows] = m.to(self.tier1b_exp_avg.dtype)\n        self.tier1b_exp_avg_sq[rows] = v.to(self.tier1b_exp_avg_sq.dtype)\n        self.tier1b_cold_weight[rows] = w.to(self.tier1b_cold_weight.dtype)\n        self._pending_tier1b_leaf = None\n        self._pending_tier1b_rows = None\n\n    def forward(\n        self,\n        input_ids: torch.LongTensor,\n        dtype: torch.dtype,\n        prev_ids: Optional[torch.LongTensor] = None,\n        hidden_states: Optional[torch.Tensor] = None,\n        pass_idx: int = 0,\n        ngram_indices: Optional[tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = None,\n    ) -> torch.Tensor:\n        param_dtype = self.project.weight.dtype\n        if ngram_indices is None:\n            ngram_indices = self.compute_ngram_indices(input_ids, self.total_buckets, prev_ids=prev_ids)\n\n        # 1. Tier 1A + 1B Lexical N-gram Lookup\n        e_lexical = self._lookup_tier1_lexical(ngram_indices, input_ids.device, param_dtype)\n        lex_hidden = self.lexical_up(e_lexical)\n\n        if hidden_states is not None:\n            h_f = hidden_states.float()\n            rms = torch.rsqrt(h_f.pow(2).mean(dim=-1, keepdim=True) + self.norm_eps)\n            h_norm_f = h_f * rms * self.norm_weight\n            h_norm = h_norm_f.to(param_dtype)\n\n            # 2. Tier 2: 100%-Active Hot Shared SwiGLU Expert\n            h_fused_in = h_norm + lex_hidden\n            shared_out = self.shared_down(F.silu(self.shared_gate(h_fused_in)) * self.shared_up(h_fused_in))\n\n            # 3. Tier 3: X-MoE Hypersphere Cosine Router + Sigmoid Affinity + ST-MoE Router Z-Loss\n            e_act = self.active_experts\n            k_act = min(self.active_top_k, e_act)\n            p_idx = min(max(int(pass_idx), 0), self.max_passes - 1)\n\n            u = F.normalize(self.router_proj(h_norm_f), p=2, dim=-1, eps=1e-6)\n            c = F.normalize(self.expert_centroids[:e_act], p=2, dim=-1, eps=1e-6)\n            temp = self.router_log_temp.exp().clamp(min=0.03, max=2.0)\n            router_logits = (torch.matmul(u, c.t()) / temp) + self.pass_router_bias[p_idx, :e_act]\n\n            if self.training:\n                log_z = torch.logsumexp(router_logits, dim=-1)\n                self.last_router_z_loss = log_z.pow(2).mean()\n            else:\n                self.last_router_z_loss = router_logits.new_zeros(())\n\n            scores = torch.sigmoid(router_logits)\n            sel_scores = scores + self.expert_bias[:e_act]\n            _, topk_idx = torch.topk(sel_scores, k=k_act, dim=-1)\n            topk_raw = torch.gather(scores, dim=-1, index=topk_idx)\n            topk_gates = (topk_raw / (topk_raw.sum(dim=-1, keepdim=True) + 1e-6)).to(param_dtype)\n\n            if self.training:\n                with torch.no_grad():\n                    one_hot = F.one_hot(topk_idx, num_classes=e_act).float()\n                    load = one_hot.sum(dim=-2).mean(dim=(0, 1)) / float(k_act)\n                    self.expert_load_ema[:e_act].lerp_(load, 0.05)\n                    target_load = 1.0 / float(e_act)\n                    self.expert_bias[:e_act].add_(0.001 * torch.sign(target_load - self.expert_load_ema[:e_act])).clamp_(-0.5, 0.5)\n\n            # Query vector q_t in R^eng_dim conditioned on both semantic hidden state and lexical N-gram\n            q = self.query_proj(h_norm) + e_lexical\n            flat_q = q.reshape(-1, self.eng_dim)\n            flat_idx = topk_idx.reshape(-1, k_act)\n            flat_gates = topk_gates.reshape(-1, k_act)\n\n            # Memory-efficient grouped expert evaluation (iterates over active experts instead of materializing [N, K, D, R])\n            routed_flat = torch.zeros((flat_q.shape[0], self.width), dtype=param_dtype, device=flat_q.device)\n            for e_id in range(e_act):\n                mask_e = (flat_idx == e_id)\n                if not mask_e.any():\n                    continue\n                token_rows, slot_cols = mask_e.nonzero(as_tuple=True)\n                q_e = flat_q.index_select(0, token_rows)\n                g_e = flat_gates[token_rows, slot_cols].unsqueeze(-1)\n                act_e = F.silu(torch.matmul(q_e, self.expert_gate[e_id])) * torch.matmul(q_e, self.expert_up[e_id])\n                out_e = torch.matmul(act_e, self.expert_down[e_id]) * g_e\n                routed_flat.index_add_(0, token_rows, out_e)\n\n            routed_out = routed_flat.view_as(shared_out)\n            combined = lex_hidden + shared_out + routed_out\n            ctx = 2.0 * torch.sigmoid(self.ctx_gate(h_norm).float()).to(param_dtype)\n            combined = combined * ctx\n        else:\n            combined = F.silu(lex_hidden)\n\n        projected = self.project(combined)\n        gate = torch.sigmoid(self.gain).to(dtype)\n        return projected.to(dtype) * gate\n\n\nclass TwoStreamMHC(nn.Module):\n    """Sublayer Dual-mHC with Pass-0/Pass-1/Pass-2 (72-Layer) Conditioning and Closed-Form 2x2 Sinkhorn."""\n    def __init__(\n        self,\n        width: int = 2560,\n        streams: int = 2,\n        iters: int = 20,\n        dynamic: bool = True,\n        sublayer_mhc: bool = True,\n        pass_adapter_rank: int = 16,\n        max_passes: int = 4,\n    ):\n        super().__init__()\n        if streams != 2:\n            raise ValueError(\'TwoStreamMHC expects 2 streams\')\n        self.width = width\n        self.streams = streams\n        self.iters = iters\n        self.dynamic = dynamic\n        self.sublayer_mhc = sublayer_mhc\n        self.pass_adapter_rank = pass_adapter_rank\n        self.max_passes = max_passes\n        self.norm_eps = 1e-6\n\n        self.read_logits = nn.Parameter(torch.tensor([2.0, -2.0], dtype=torch.float32))\n        self.write_logits = nn.Parameter(torch.tensor([2.0, -2.0], dtype=torch.float32))\n        self.mix_logits = nn.Parameter(torch.tensor([[2.0, -2.0], [-2.0, 2.0]], dtype=torch.float32))\n        self.extra_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))\n        self.p3_extra_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))\n\n        self.ffn_read_logits = nn.Parameter(torch.tensor([2.0, -2.0], dtype=torch.float32))\n        self.ffn_write_logits = nn.Parameter(torch.tensor([2.0, -2.0], dtype=torch.float32))\n        self.ffn_mix_logits = nn.Parameter(torch.tensor([[2.0, -2.0], [-2.0, 2.0]], dtype=torch.float32))\n        self.ffn_extra_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))\n        self.p3_ffn_extra_gain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))\n\n        self.pass_routing_bias = nn.Parameter(torch.zeros(max_passes, 2, 8, dtype=torch.float32))\n\n        if dynamic:\n            flat_dim = streams * width\n            self.norm_weight = nn.Parameter(torch.ones(flat_dim, dtype=torch.float32))\n            self.dynamic_proj = nn.Linear(flat_dim, 8, bias=False, dtype=torch.float32)\n            nn.init.zeros_(self.dynamic_proj.weight)\n            self.ffn_norm_weight = nn.Parameter(torch.ones(flat_dim, dtype=torch.float32))\n            self.ffn_dynamic_proj = nn.Linear(flat_dim, 8, bias=False, dtype=torch.float32)\n            nn.init.zeros_(self.ffn_dynamic_proj.weight)\n\n        if pass_adapter_rank > 0:\n            self.pass_down = nn.ModuleList([\n                nn.Linear(width, pass_adapter_rank, bias=False, dtype=torch.float32)\n                for _ in range(max_passes)\n            ])\n            self.pass_up = nn.ModuleList([\n                nn.Linear(pass_adapter_rank, width, bias=False, dtype=torch.float32)\n                for _ in range(max_passes)\n            ])\n            for p_idx in range(max_passes):\n                nn.init.kaiming_uniform_(self.pass_down[p_idx].weight, a=math.sqrt(5))\n                nn.init.zeros_(self.pass_up[p_idx].weight)\n\n    def mixing_matrix(self, dynamic_delta: Optional[torch.Tensor] = None, base_logits: Optional[torch.Tensor] = None) -> torch.Tensor:\n        base = self.mix_logits if base_logits is None else base_logits\n        log_p = base if dynamic_delta is None else (base + dynamic_delta)\n        log_p = log_p.clamp(min=-30.0, max=30.0)\n        # Closed-form exact 2x2 symmetric Birkhoff projection:\n        # For a 2x2 matrix [[a, b], [c, d]], Sinkhorn converges to a symmetric doubly-stochastic\n        # matrix parameterized by the cross-ratio (a + d) - (b + c):\n        diag_adv = 0.5 * ((log_p[..., 0, 0] + log_p[..., 1, 1]) - (log_p[..., 0, 1] + log_p[..., 1, 0]))\n        p_keep = torch.sigmoid(diag_adv)\n        p_swap = 1.0 - p_keep\n        row0 = torch.stack([p_keep, p_swap], dim=-1)\n        row1 = torch.stack([p_swap, p_keep], dim=-1)\n        return torch.stack([row0, row1], dim=-2)\n\n    def _compute_routing(self, streams: torch.Tensor, is_ffn: bool, pass_idx: int = 0):\n        p_idx = min(max(pass_idx, 0), self.max_passes - 1)\n        r_base = self.ffn_read_logits if is_ffn else self.read_logits\n        w_base = self.ffn_write_logits if is_ffn else self.write_logits\n        m_base = self.ffn_mix_logits if is_ffn else self.mix_logits\n        p_bias = self.pass_routing_bias[p_idx, 1 if is_ffn else 0]\n        r_base = r_base + p_bias[0:2]\n        w_base = w_base + p_bias[2:4]\n        m_base = m_base + p_bias[4:8].view(2, 2)\n\n        if not self.dynamic:\n            p = self.mixing_matrix(base_logits=m_base).to(streams.dtype)\n            r = F.softmax(r_base, dim=0).to(streams.dtype)\n            w = (2.0 * torch.sigmoid(w_base)).to(streams.dtype)\n            return p, r, w, True\n\n        flat = streams.permute(1, 2, 0, 3).reshape(streams.shape[1], streams.shape[2], -1).float()\n        rms = torch.rsqrt(flat.pow(2).mean(dim=-1, keepdim=True) + self.norm_eps)\n        nw = self.ffn_norm_weight if is_ffn else self.norm_weight\n        proj = self.ffn_dynamic_proj if is_ffn else self.dynamic_proj\n        raw = proj(flat * rms * nw)\n        r = F.softmax(r_base + raw[..., 0:2], dim=-1).to(streams.dtype)\n        w = (2.0 * torch.sigmoid(w_base + raw[..., 2:4])).to(streams.dtype)\n        p = self.mixing_matrix(raw[..., 4:8].view(*raw.shape[:-1], 2, 2), base_logits=m_base).to(streams.dtype)\n        return p, r, w, False\n\n    @staticmethod\n    def _mix_and_read(streams: torch.Tensor, p: torch.Tensor, r: torch.Tensor, is_static: bool):\n        if is_static:\n            mixed = torch.einsum(\'ij,jbtd->ibtd\', p, streams)\n            layer_in = torch.einsum(\'i,ibtd->btd\', r, mixed)\n        else:\n            mixed = torch.einsum(\'btij,jbtd->ibtd\', p, streams)\n            layer_in = torch.einsum(\'bti,ibtd->btd\', r, mixed)\n        return mixed, layer_in\n\n    @staticmethod\n    def _write(mixed: torch.Tensor, delta: torch.Tensor, w: torch.Tensor, is_static: bool):\n        if is_static:\n            return mixed + w.view(2, 1, 1, 1) * delta.unsqueeze(0)\n        return mixed + w.permute(2, 0, 1).unsqueeze(-1) * delta.unsqueeze(0)\n\n    def _apply_pass_adapter(self, h: torch.Tensor, pass_idx: int) -> torch.Tensor:\n        if self.pass_adapter_rank <= 0 or pass_idx <= 0:\n            return h\n        p_idx = min(pass_idx, self.max_passes - 1)\n        delta = self.pass_up[p_idx](F.silu(self.pass_down[p_idx](h.float())))\n        return h + delta.to(h.dtype)\n\n    def forward(self, streams: torch.Tensor, layer: nn.Module, position_embeddings, attention_mask,\n                extra: bool = False, pass_idx: int = 0, past_key_values=None, cache_position=None) -> torch.Tensor:\n        p_attn, r_attn, w_attn, static_attn = self._compute_routing(streams, is_ffn=False, pass_idx=pass_idx)\n        mixed_attn, attn_in = self._mix_and_read(streams, p_attn, r_attn, static_attn)\n\n        if self.sublayer_mhc and hasattr(layer, \'self_attn\') and hasattr(layer, \'mlp\'):\n            target_dtype = layer.mlp.gate_proj.weight.dtype\n            normed_attn = layer.input_layernorm(attn_in).to(target_dtype)\n            attn_out, _ = layer.self_attn(\n                hidden_states=normed_attn,\n                position_embeddings=position_embeddings,\n                attention_mask=attention_mask,\n                past_key_values=past_key_values,\n                cache_position=cache_position,\n            )\n            attn_delta = attn_out.to(streams.dtype)\n            if extra:\n                gain_param = self.p3_extra_gain if pass_idx >= 2 else self.extra_gain\n                attn_delta = attn_delta * torch.tanh(gain_param.float()).to(streams.dtype)\n            mid_streams = self._write(mixed_attn, attn_delta, w_attn, static_attn)\n\n            p_ffn, r_ffn, w_ffn, static_ffn = self._compute_routing(mid_streams, is_ffn=True, pass_idx=pass_idx)\n            mixed_ffn, ffn_in = self._mix_and_read(mid_streams, p_ffn, r_ffn, static_ffn)\n            ffn_in = self._apply_pass_adapter(ffn_in, pass_idx)\n            normed_ffn = layer.post_attention_layernorm(ffn_in).to(target_dtype)\n            ffn_out = layer.mlp(normed_ffn)\n            ffn_delta = ffn_out.to(streams.dtype)\n            if extra:\n                fgain_param = self.p3_ffn_extra_gain if pass_idx >= 2 else self.ffn_extra_gain\n                ffn_delta = ffn_delta * torch.tanh(fgain_param.float()).to(streams.dtype)\n            return self._write(mixed_ffn, ffn_delta, w_ffn, static_ffn)\n\n        layer_in = self._apply_pass_adapter(attn_in, pass_idx)\n        layer_out = layer(\n            layer_in,\n            position_embeddings=position_embeddings,\n            attention_mask=attention_mask,\n            past_key_values=past_key_values,\n            cache_position=cache_position,\n        )\n        delta = (layer_out - attn_in).to(streams.dtype)\n        if extra:\n            gain_param = self.p3_extra_gain if pass_idx >= 2 else self.extra_gain\n            delta = delta * torch.tanh(gain_param.float()).to(streams.dtype)\n        return self._write(mixed_attn, delta, w_attn, static_attn)\n\n\ndef fused_masked_cross_entropy(\n    hidden_states: torch.Tensor,\n    lm_head_weight: torch.Tensor,\n    labels: torch.Tensor,\n    chunk_tokens: int = 512,\n) -> torch.Tensor:\n    shift_h = hidden_states[:, :-1, :].contiguous()\n    shift_labels = labels[:, 1:].contiguous()\n    flat_h = shift_h.view(-1, shift_h.shape[-1])\n    flat_labels = shift_labels.view(-1)\n    valid_mask = flat_labels != -100\n    valid_indices = valid_mask.nonzero(as_tuple=False).squeeze(-1)\n    if valid_indices.numel() == 0:\n        return (flat_h[:1].float().sum() * 0.0)\n    valid_h = flat_h.index_select(0, valid_indices)\n    valid_t = flat_labels.index_select(0, valid_indices)\n    w = lm_head_weight.to(valid_h.dtype)\n    total_loss = valid_h.new_zeros((), dtype=torch.float32)\n    n_valid = valid_t.numel()\n    for start in range(0, n_valid, max(64, chunk_tokens)):\n        end = min(start + max(64, chunk_tokens), n_valid)\n        logits_chunk = F.linear(valid_h[start:end], w).float()\n        total_loss = total_loss + F.cross_entropy(logits_chunk, valid_t[start:end], reduction=\'sum\')\n    return total_loss / float(n_valid)\n\n\nclass SparkXLoopV03(nn.Module):\n    """Spark-X2.6-4B-En4B-A1B (v0.3):\n    - 36 physical layers unrolled up to 72 effective layers via 3-Pass Telescoping Recurrence:\n      Prelude [0..4) + Pass-1 [4..28) + Pass-2 [4..28) + Pass-3 [12..24) + Coda [28..36) = 72 virtual layers.\n    - 8 Macro-Block MoE-Engram Anchors at physical layers 3, 7, 11, 15, 19, 23, 27, 31\n      (500M total / 125M active per anchor = 4.0B Total / 1.0B Active Engram).\n    """\n    def __init__(self, base_model: nn.Module, settings: Optional[LoopV03Settings] = None,\n                 strict: bool = True, patch_sdpa: bool = True):\n        super().__init__()\n        self.base = base_model\n        self.settings = settings or LoopV03Settings()\n        if patch_sdpa:\n            patch_spark_attention_for_long_context(self.base)\n        self.n_layers = len(self.base.model.layers)\n        if strict and self.n_layers != 36:\n            raise ValueError(f\'Expected 36 Spark layers, got {self.n_layers}\')\n        width = int(self.base.config.hidden_size)\n        if strict and width != 2560:\n            raise ValueError(f\'Expected hidden_size 2560, got {width}\')\n\n        self.loop_start = self.settings.start\n        self.loop_end = self.settings.end\n        self.loop_passes = self.settings.passes\n        self.p3_start = self.settings.p3_start\n        self.p3_end = self.settings.p3_end\n        self.enable_p3 = self.settings.enable_p3\n        self.mhc_enabled = self.settings.use_mhc\n        self.engram_enabled = self.settings.use_engram\n        self.enable_checkpoint = self.settings.checkpoint_layers\n        self.offload_checkpoint_to_cpu = self.settings.offload_checkpoint_to_cpu\n\n        self._validate_loop()\n        self.controllers = nn.ModuleDict({\n            str(i): TwoStreamMHC(\n                width=width,\n                streams=self.settings.mhc_streams,\n                dynamic=self.settings.dynamic_mhc,\n                sublayer_mhc=self.settings.sublayer_mhc,\n                pass_adapter_rank=self.settings.pass_adapter_rank,\n                max_passes=self.settings.max_passes,\n            )\n            for i in range(self.loop_start, self.loop_end)\n        })\n\n        # 8 Macro-Block Anchors: Layer 3 (Prelude) + Layers 7, 11, 15, 19, 23, 27 (Recurrent Core) + Layer 31 (Coda)\n        param_dtype = self.base.model.embedding.weight.dtype\n        if param_dtype not in (torch.bfloat16, torch.float16):\n            param_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32\n\n        def _make_engram():\n            return ZipfianTieredMoEEngram(\n                hot_buckets=self.settings.eng_hot_buckets,\n                cold_buckets=self.settings.eng_cold_buckets,\n                eng_dim=self.settings.eng_dim,\n                width=width,\n                shared_rank=self.settings.eng_shared_rank,\n                num_experts=self.settings.eng_experts,\n                initial_active_experts=self.settings.eng_initial_active_experts,\n                top_k=self.settings.eng_top_k,\n                initial_top_k=self.settings.eng_initial_top_k,\n                expert_rank=self.settings.eng_expert_rank,\n                route_dim=self.settings.eng_route_dim,\n                max_passes=self.settings.max_passes,\n                offload_tier1b_cpu=self.settings.eng_offload_tier1b_cpu,\n                param_dtype=param_dtype,\n            )\n\n        self.engram = _make_engram()  # Anchor 0: Layer 3 (end of Prelude [0..4))\n        stride = max(1, int(self.settings.backbone_engram_stride))\n        # Core + Coda macro-block anchors: Layers 7, 11, 15, 19, 23, 27, 31 (7 anchors + 1 Prelude = 8 anchors total)\n        anchor_indices = [\n            idx for idx in range(self.loop_start + stride - 1, min(self.n_layers - 4, 32), stride)\n        ]\n        self.backbone_engrams = nn.ModuleDict({\n            str(idx): _make_engram() for idx in anchor_indices\n        })\n\n    def all_engram_modules(self) -> list[ZipfianTieredMoEEngram]:\n        return [self.engram] + [mod for mod in self.backbone_engrams.values()]\n\n    def upcycle_all_engrams_8_to_32(self, noise_std: float = 1e-3):\n        for mod in self.all_engram_modules():\n            mod.upcycle_experts_8_to_32(noise_std=noise_std)\n\n    def step_all_tier1b_sparse_adam(self, lr: float = 2e-4):\n        for mod in self.all_engram_modules():\n            mod.step_tier1b_sparse_adam(lr=lr)\n\n    def total_router_z_loss(self) -> torch.Tensor:\n        losses = [mod.last_router_z_loss for mod in self.all_engram_modules() if mod.last_router_z_loss is not None]\n        if not losses:\n            return torch.tensor(0.0, device=self._head_weight().device)\n        return torch.stack([l.to(self._head_weight().device) for l in losses]).mean()\n\n    def _validate_loop(self):\n        if not (0 <= self.loop_start < self.loop_end <= self.n_layers):\n            raise ValueError(\'Invalid loop boundaries\')\n        if (self.loop_start % 4) or (self.loop_end % 4):\n            raise ValueError(\'Loop boundaries must align to Spark groups of four\')\n        if self.enable_p3:\n            if not (self.loop_start <= self.p3_start < self.p3_end <= self.loop_end):\n                raise ValueError(\'Pass-3 inner core must lie inside [loop_start, loop_end]\')\n            if (self.p3_start % 4) or (self.p3_end % 4):\n                raise ValueError(\'Pass-3 boundaries must align to Spark groups of four\')\n\n    @property\n    def execution_plan(self) -> list[int]:\n        plan = (\n            list(range(0, self.loop_start))\n            + list(range(self.loop_start, self.loop_end)) * self.loop_passes\n        )\n        if self.enable_p3 and self.loop_passes >= 2:\n            plan = plan + list(range(self.p3_start, self.p3_end))\n        plan = plan + list(range(self.loop_end, self.n_layers))\n        return plan\n\n    @property\n    def total_virtual_layers(self) -> int:\n        return len(self.execution_plan)\n\n    def effective_depth(self) -> int:\n        return len(self.execution_plan)\n\n    def _make_loop_cache_config(self):\n        plan = tuple(self.execution_plan)\n        if getattr(self, \'_cached_loop_plan\', None) == plan and getattr(self, \'_cached_loop_cfg\', None) is not None:\n            return self._cached_loop_cfg\n        cfg = copy.deepcopy(self.base.config)\n        cfg.num_hidden_layers = len(plan)\n        if hasattr(cfg, \'layer_types\') and cfg.layer_types:\n            cfg.layer_types = [cfg.layer_types[i] for i in plan]\n        self._cached_loop_plan = plan\n        self._cached_loop_cfg = cfg\n        return cfg\n\n    def _checkpoint_context(self):\n        if self.training and self.enable_checkpoint and self.offload_checkpoint_to_cpu:\n            return torch.autograd.graph.save_on_cpu(pin_memory=True)\n        return nullcontext()\n\n    def set_loop(\n        self,\n        start: int,\n        end: int,\n        passes: int,\n        use_mhc: bool = True,\n        use_engram: bool = True,\n        enable_p3: bool = False,\n        p3_start: int = 12,\n        p3_end: int = 24,\n    ):\n        self.loop_start, self.loop_end, self.loop_passes = start, end, passes\n        self.mhc_enabled, self.engram_enabled = use_mhc, use_engram\n        self.enable_p3, self.p3_start, self.p3_end = enable_p3, p3_start, p3_end\n        self._cached_loop_cfg = None\n        self._validate_loop()\n        for idx in range(start, end):\n            if str(idx) not in self.controllers:\n                raise RuntimeError(f\'Controller for layer {idx} was not created\')\n\n    def _forward_context(self, embeds, attention_mask, past_key_values=None, cache_position=None):\n        model = self.base.model\n        cfg = self.base.config\n        device = embeds.device\n        if cache_position is None:\n            past_seen = past_key_values.get_seq_length(0) if past_key_values is not None else 0\n            cache_position = torch.arange(past_seen, past_seen + embeds.shape[1], device=device)\n        position_ids = cache_position.unsqueeze(0)\n        if hasattr(model, \'make_loop_context\'):\n            return model.make_loop_context(embeds, attention_mask), cache_position\n        if past_key_values is not None and embeds.shape[1] == 1 and attention_mask is None:\n            masks = {\'full_attention\': None, \'sliding_attention\': None}\n        else:\n            from transformers.masking_utils import create_causal_mask, create_sliding_window_causal_mask\n            mask_cfg = self._make_loop_cache_config() if past_key_values is not None else cfg\n            kwargs = dict(config=mask_cfg, input_embeds=embeds, attention_mask=attention_mask,\n                          cache_position=cache_position, past_key_values=past_key_values, position_ids=position_ids)\n            masks = {\'full_attention\': create_causal_mask(**kwargs)}\n            if getattr(model, \'has_sliding_layers\', False):\n                masks[\'sliding_attention\'] = create_sliding_window_causal_mask(**kwargs)\n        implementation = importlib.import_module(type(model.layers[0].self_attn).__module__)\n        rope = {}\n        rope_dtype = embeds.dtype if (past_key_values is not None and embeds.dtype in (torch.bfloat16, torch.float16)) else None\n        for layer_type in set(cfg.layer_types):\n            cos, sin = implementation.compute_rope_cos_sin(\n                cache_position, cfg.head_dim, cfg.get_rope_theta(layer_type),\n                partial_rotary_factor=cfg.get_partial_rotary_factor(layer_type), device=device,\n            )\n            if rope_dtype is not None:\n                cos = cos.to(rope_dtype)\n                sin = sin.to(rope_dtype)\n            rope[layer_type] = (cos, sin)\n        return (masks, rope), cache_position\n\n    def _apply_layer(self, idx, h, masks, ropes, past_key_values=None, cache_position=None, virtual_idx: Optional[int] = None):\n        layer = self.base.model.layers[idx]\n        kind = layer.layer_type\n        rp, mk = ropes[kind], masks[kind]\n        kv_dtype = self.base.model.embedding.weight.dtype\n        v_cache = (\n            _VisitCache(past_key_values, idx, virtual_idx if virtual_idx is not None else idx, kv_dtype=kv_dtype)\n            if past_key_values is not None else None\n        )\n        target_dtype = layer.mlp.gate_proj.weight.dtype\n        def step(x, _layer=layer, _rp=rp, _mk=mk, _vc=v_cache, _cp=cache_position, _dt=target_dtype):\n            x_in = x.to(_dt) if x.dtype != _dt else x\n            out = _layer(x_in, position_embeddings=_rp, attention_mask=_mk,\n                         past_key_values=_vc, cache_position=_cp)\n            return out.to(x.dtype) if out.dtype != x.dtype else out\n        if v_cache is None and self.training and self.enable_checkpoint and (\n            h.requires_grad or any(p.requires_grad for p in layer.parameters())\n        ):\n            with self._checkpoint_context():\n                return checkpoint(step, h, use_reentrant=False)\n        return step(h)\n\n    def forward_hidden(self, input_ids: torch.LongTensor, attention_mask=None,\n                       past_key_values=None, cache_position=None,\n                       prev_ids: Optional[torch.LongTensor] = None) -> torch.Tensor:\n        if input_ids.ndim != 2:\n            raise ValueError(\'Expected input_ids shape [batch, sequence]\')\n        # Ahead-of-time deterministic token-ID hashing (computed before Layer 0!)\n        ngram_indices = (\n            self.engram.compute_ngram_indices(input_ids, self.engram.total_buckets, prev_ids=prev_ids)\n            if self.engram_enabled else None\n        )\n        embeds = self.base.model.embedding(input_ids)\n        (masks, ropes), cache_position = self._forward_context(\n            embeds, attention_mask, past_key_values=past_key_values, cache_position=cache_position\n        )\n        h = embeds.float()\n        kv_dtype = embeds.dtype\n        virtual_idx = 0\n\n        # 1. Prelude layers [0..loop_start)\n        for idx in range(self.loop_start):\n            h = self._apply_layer(idx, h, masks, ropes, past_key_values=past_key_values,\n                                  cache_position=cache_position, virtual_idx=virtual_idx)\n            virtual_idx += 1\n        if self.engram_enabled:\n            h = h + self.engram(\n                input_ids, h.dtype, prev_ids=prev_ids, hidden_states=h,\n                pass_idx=0, ngram_indices=ngram_indices,\n            )\n\n        # Build pass schedule: Pass 0..loop_passes-1 over [loop_start, loop_end), plus optional Pass 2 over [p3_start, p3_end)\n        pass_spans = [(p, self.loop_start, self.loop_end) for p in range(self.loop_passes)]\n        if self.enable_p3 and self.loop_passes >= 2:\n            pass_spans.append((2, self.p3_start, self.p3_end))\n\n        if not self.mhc_enabled:\n            for p, s_idx, e_idx in pass_spans:\n                for idx in range(s_idx, e_idx):\n                    h = self._apply_layer(idx, h, masks, ropes, past_key_values=past_key_values,\n                                          cache_position=cache_position, virtual_idx=virtual_idx)\n                    if self.engram_enabled and str(idx) in self.backbone_engrams:\n                        eng_delta = self.backbone_engrams[str(idx)](\n                            input_ids, h.dtype, prev_ids=prev_ids, hidden_states=h,\n                            pass_idx=p, ngram_indices=ngram_indices,\n                        )\n                        if p == 0:\n                            h = h + eng_delta\n                    virtual_idx += 1\n        else:\n            streams = torch.stack((h, h), dim=0)\n            for p, s_idx, e_idx in pass_spans:\n                for idx in range(s_idx, e_idx):\n                    kind = self.base.model.layers[idx].layer_type\n                    layer = self.base.model.layers[idx]\n                    ctrl = self.controllers[str(idx)]\n                    rp, mk = ropes[kind], masks[kind]\n                    v_cache = _VisitCache(past_key_values, idx, virtual_idx, kv_dtype=kv_dtype) if past_key_values is not None else None\n                    eng_mod = (\n                        self.backbone_engrams[str(idx)]\n                        if (self.engram_enabled and str(idx) in self.backbone_engrams)\n                        else None\n                    )\n                    virtual_idx += 1\n                    def step(s, _ctrl=ctrl, _layer=layer, _rp=rp, _mk=mk, _extra=(p > 0), _p=p, _vc=v_cache, _cp=cache_position, _eng=eng_mod):\n                        s_out = _ctrl(s, _layer, _rp, _mk, extra=_extra, pass_idx=_p,\n                                      past_key_values=_vc, cache_position=_cp)\n                        if _eng is not None:\n                            eng_delta = _eng(\n                                input_ids, s_out.dtype, prev_ids=prev_ids, hidden_states=s_out[0],\n                                pass_idx=_p, ngram_indices=ngram_indices,\n                            )\n                            if _extra:\n                                fgain = _ctrl.p3_ffn_extra_gain if _p >= 2 else _ctrl.ffn_extra_gain\n                                eng_delta = eng_delta * torch.tanh(fgain.float()).to(s_out.dtype)\n                            s_out = torch.stack((s_out[0] + eng_delta, s_out[1]), dim=0)\n                        return s_out\n                    if v_cache is None and self.training and self.enable_checkpoint:\n                        with self._checkpoint_context():\n                            streams = checkpoint(step, streams, use_reentrant=False)\n                    else:\n                        streams = step(streams)\n            h = streams[0]\n\n        # 3. Coda layers [loop_end..n_layers) (includes Anchor 7 at Layer 31!)\n        for idx in range(self.loop_end, self.n_layers):\n            h = self._apply_layer(idx, h, masks, ropes, past_key_values=past_key_values,\n                                  cache_position=cache_position, virtual_idx=virtual_idx)\n            if self.engram_enabled and str(idx) in self.backbone_engrams:\n                h = h + self.backbone_engrams[str(idx)](\n                    input_ids, h.dtype, prev_ids=prev_ids, hidden_states=h,\n                    pass_idx=0, ngram_indices=ngram_indices,\n                )\n            virtual_idx += 1\n        norm_in = h.to(kv_dtype) if (not self.training and kv_dtype in (torch.bfloat16, torch.float16)) else h\n        return self.base.model.norm(norm_in).to(kv_dtype)\n\n    def _head_weight(self) -> torch.Tensor:\n        if getattr(self.base.config, \'tie_word_embeddings\', False):\n            return self.base.model.embedding.weight\n        return self.base.lm_head.weight\n\n    def forward_logits(self, input_ids: torch.LongTensor, attention_mask=None,\n                       past_key_values=None, cache_position=None,\n                       prev_ids: Optional[torch.LongTensor] = None):\n        h = self.forward_hidden(\n            input_ids, attention_mask=attention_mask,\n            past_key_values=past_key_values, cache_position=cache_position,\n            prev_ids=prev_ids,\n        )\n        if getattr(self.base.config, \'tie_word_embeddings\', False):\n            return F.linear(h, self.base.model.embedding.weight)\n        return self.base.lm_head(h)\n\n    def forward(self, input_ids, labels=None, attention_mask=None, return_logits: bool = False):\n        if labels is None:\n            logits = self.forward_logits(input_ids, attention_mask)\n            return {\'logits\': logits}\n        if input_ids.shape[1] < 2:\n            raise ValueError(\'Need at least 2 tokens for next-token objective\')\n        h = self.forward_hidden(input_ids, attention_mask=attention_mask)\n        ce_loss = fused_masked_cross_entropy(\n            h, self._head_weight(), labels, chunk_tokens=self.settings.ce_chunk_tokens\n        )\n        z_loss = self.total_router_z_loss() * float(self.settings.eng_z_loss_coeff) if (self.training and self.engram_enabled) else 0.0\n        loss = ce_loss + z_loss\n        if return_logits:\n            logits = F.linear(h, self._head_weight()) if getattr(self.base.config, \'tie_word_embeddings\', False) else self.base.lm_head(h)\n            return {\'logits\': logits, \'loss\': loss, \'ce_loss\': ce_loss, \'z_loss\': z_loss}\n        return {\'logits\': None, \'loss\': loss, \'ce_loss\': ce_loss, \'z_loss\': z_loss}\n\n    @torch.no_grad()\n    def generate_cached(self, input_ids, max_new_tokens=256, eos_token_id=None):\n        from transformers.cache_utils import DynamicCache\n        prev_train = self.training\n        self.eval()\n        try:\n            cache = DynamicCache(config=self._make_loop_cache_config())\n        except TypeError:\n            cache = DynamicCache()\n        out = input_ids\n        try:\n            logits = self.forward_logits(out, past_key_values=cache)\n            next_id = logits[:, -1].argmax(-1, keepdim=True)\n            prev_tok = out[:, -2:]\n            out = torch.cat((out, next_id), dim=-1)\n            if eos_token_id is not None and torch.all(next_id == eos_token_id):\n                return out\n            for _ in range(max_new_tokens - 1):\n                logits = self.forward_logits(next_id, past_key_values=cache, prev_ids=prev_tok)\n                prev_tok = out[:, -2:]\n                next_id = logits[:, -1].argmax(-1, keepdim=True)\n                out = torch.cat((out, next_id), dim=-1)\n                if eos_token_id is not None and torch.all(next_id == eos_token_id):\n                    break\n        finally:\n            self.train(prev_train)\n        return out\n\n\ndef set_trainables_v03(model: SparkXLoopV03, full_finetune: bool = True, freeze_backbone: bool = False):\n    for p in model.base.parameters():\n        p.requires_grad_(bool(full_finetune and not freeze_backbone))\n    # Always keep embeddings frozen to protect 167,168 vocabulary geometry\n    for p in model.base.model.embedding.parameters():\n        p.requires_grad_(False)\n    if full_finetune and not freeze_backbone:\n        for p in model.base.model.norm.parameters():\n            p.requires_grad_(True)\n    for p in model.controllers.parameters():\n        p.requires_grad_(True)\n    for p in model.engram.parameters():\n        p.requires_grad_(True)\n    for p in model.backbone_engrams.parameters():\n        p.requires_grad_(True)\n    return {n: p for n, p in model.named_parameters() if p.requires_grad}\n\n\ndef build_optimizer_v03(\n    model: SparkXLoopV03,\n    lr_adapter: float = 2.0e-4,\n    lr_full: float = 2.0e-6,\n    lr_mhc: float = 5.0e-3,\n    weight_decay: float = 0.01,\n    use_8bit: bool = True,\n):\n    adapters, routing, backbone = [], [], []\n    for name, p in model.named_parameters():\n        if not p.requires_grad:\n            continue\n        if (\n            (name.startswith(\'controllers.\') and \'.pass_\' not in name)\n            or name.endswith(\'.gain\')\n            or name.endswith(\'.router_log_temp\')\n            or name.endswith(\'.scale_weights\')\n        ):\n            routing.append(p)\n        elif name.startswith(\'base.model.\'):\n            backbone.append(p)\n        else:\n            adapters.append(p)\n    groups = [\n        {\'params\': adapters, \'lr\': lr_adapter, \'weight_decay\': weight_decay},\n        {\'params\': routing, \'lr\': lr_mhc, \'weight_decay\': 0.0},\n    ]\n    if backbone:\n        groups.append({\'params\': backbone, \'lr\': lr_full, \'weight_decay\': weight_decay})\n    if use_8bit:\n        try:\n            import bitsandbytes as bnb\n            return bnb.optim.PagedAdamW8bit(groups, betas=(0.9, 0.95), eps=1e-8)\n        except Exception:\n            pass\n    if torch.cuda.is_available():\n        try:\n            return torch.optim.AdamW(groups, betas=(0.9, 0.95), eps=1e-8, fused=True)\n        except Exception:\n            pass\n    return torch.optim.AdamW(groups, betas=(0.9, 0.95), eps=1e-8)\n\n\ndef count_en4b_a1b_params(model: SparkXLoopV03) -> dict[str, int]:\n    """Compute exact parameter breakdown across Backbone, GPU Engrams, and Offloaded Tier-1B DRAM/SSD."""\n    backbone_total = sum(p.numel() for p in model.base.parameters())\n    ctrl_total = sum(p.numel() for p in model.controllers.parameters())\n    eng_gpu_total = sum(p.numel() for mod in model.all_engram_modules() for p in mod.parameters())\n    eng_cpu_tier1b = sum(\n        mod.tier1b_cold_weight.numel()\n        for mod in model.all_engram_modules()\n        if getattr(mod, \'tier1b_cold_weight\', None) is not None\n    )\n    eng_total = eng_gpu_total + eng_cpu_tier1b\n    # Active Engram per pass across 8 anchors: Tier-2 (100%) + Tier-3 (active_top_k / num_experts) + ctx_gate/project/query\n    active_eng = 0\n    for mod in model.all_engram_modules():\n        t2 = mod.shared_gate.weight.numel() + mod.shared_up.weight.numel() + mod.shared_down.weight.numel()\n        per_exp = (mod.eng_dim * mod.expert_rank * 2) + (mod.expert_rank * mod.width)\n        t3_act = mod.active_top_k * per_exp\n        proj_act = mod.lexical_up.weight.numel() + mod.query_proj.weight.numel() + mod.ctx_gate.weight.numel() + mod.project.weight.numel()\n        active_eng += (t2 + t3_act + proj_act)\n    return {\n        \'backbone_total\': backbone_total,\n        \'controllers_total\': ctrl_total,\n        \'engram_gpu_total\': eng_gpu_total,\n        \'engram_cpu_tier1b_total\': eng_cpu_tier1b,\n        \'engram_total\': eng_total,\n        \'engram_active_per_pass\': active_eng,\n        \'model_total_params\': backbone_total + ctrl_total + eng_total,\n        \'model_active_params\': backbone_total + ctrl_total + active_eng,\n    }\n\n\ndef loop_v03_state_dict(model: SparkXLoopV03, include_tier1b: bool = True) -> dict:\n    """Serialize all v0.3 loop modules (mHC controllers + 8 MoE-Engrams + optional offloaded Tier-1B banks)."""\n    tier1b = {}\n    if include_tier1b:\n        if getattr(model.engram, \'tier1b_cold_weight\', None) is not None:\n            tier1b[\'engram\'] = model.engram.tier1b_cold_weight.detach().cpu()\n        for k, mod in model.backbone_engrams.items():\n            if getattr(mod, \'tier1b_cold_weight\', None) is not None:\n                tier1b[f\'backbone_engrams.{k}\'] = mod.tier1b_cold_weight.detach().cpu()\n    active_meta = {\n        \'engram_active_experts\': model.engram.active_experts,\n        \'engram_active_top_k\': model.engram.active_top_k,\n    }\n    return {\n        \'version\': \'v0.3-en4b-a1b\',\n        \'settings\': dataclasses.asdict(model.settings),\n        \'active_meta\': active_meta,\n        \'controllers\': {k: v.detach().cpu() for k, v in model.controllers.state_dict().items()},\n        \'engram\': {k: v.detach().cpu() for k, v in model.engram.state_dict().items()},\n        \'backbone_engrams\': {k: v.detach().cpu() for k, v in model.backbone_engrams.state_dict().items()},\n        \'tier1b_cold\': tier1b,\n    }\n\n\ndef load_loop_v03_state_dict(model: SparkXLoopV03, state: dict, strict: bool = False) -> dict:\n    """Restore v0.3 loop state (including active expert count and pinned Host DRAM Tier-1B weights)."""\n    meta = state.get(\'active_meta\', {})\n    act_e = int(meta.get(\'engram_active_experts\', model.engram.active_experts))\n    act_k = int(meta.get(\'engram_active_top_k\', model.engram.active_top_k))\n    for mod in model.all_engram_modules():\n        mod.active_experts = min(act_e, mod.num_experts)\n        mod.active_top_k = min(act_k, mod.active_experts)\n    c_res = model.controllers.load_state_dict(state[\'controllers\'], strict=strict)\n    e_res = model.engram.load_state_dict(state[\'engram\'], strict=strict)\n    b_res = model.backbone_engrams.load_state_dict(state[\'backbone_engrams\'], strict=strict)\n    tier1b = state.get(\'tier1b_cold\', {})\n    if \'engram\' in tier1b and getattr(model.engram, \'tier1b_cold_weight\', None) is not None:\n        model.engram.tier1b_cold_weight.copy_(tier1b[\'engram\'])\n    for k, mod in model.backbone_engrams.items():\n        key = f\'backbone_engrams.{k}\'\n        if key in tier1b and getattr(mod, \'tier1b_cold_weight\', None) is not None:\n            mod.tier1b_cold_weight.copy_(tier1b[key])\n    return {\'controllers\': c_res, \'engram\': e_res, \'backbone_engrams\': b_res}\n\n\n@torch.no_grad()\ndef warm_start_from_v02_state(model: SparkXLoopV03, v02_state: dict) -> dict[str, int]:\n    """Warm-start v0.3 mHC controllers and compatible Engram gates/norms from a v0.2 `loop_state.pt`."""\n    copied_ctrl, copied_eng = 0, 0\n    v02_ctrl = v02_state.get(\'controllers\', {})\n    cur_ctrl = model.controllers.state_dict()\n    for k, src_t in v02_ctrl.items():\n        if k in cur_ctrl:\n            dst_t = cur_ctrl[k]\n            if dst_t.shape == src_t.shape:\n                dst_t.copy_(src_t.to(device=dst_t.device, dtype=dst_t.dtype))\n                copied_ctrl += 1\n            elif dst_t.ndim == src_t.ndim and dst_t.shape[1:] == src_t.shape[1:] and dst_t.shape[0] >= src_t.shape[0]:\n                dst_t[:src_t.shape[0]].copy_(src_t.to(device=dst_t.device, dtype=dst_t.dtype))\n                copied_ctrl += 1\n\n    def _copy_compatible_engram(dst_mod: ZipfianTieredMoEEngram, src_dict: dict) -> int:\n        cnt = 0\n        dst_sd = dst_mod.state_dict()\n        for k in (\'scale_weights\', \'norm_weight\', \'ctx_gate.weight\', \'ctx_gate.bias\', \'project.weight\', \'gain\'):\n            if k in src_dict and k in dst_sd and dst_sd[k].shape == src_dict[k].shape:\n                dst_sd[k].copy_(src_dict[k].to(device=dst_sd[k].device, dtype=dst_sd[k].dtype))\n                cnt += 1\n        return cnt\n\n    if \'engram\' in v02_state:\n        copied_eng += _copy_compatible_engram(model.engram, v02_state[\'engram\'])\n    if \'backbone_engrams\' in v02_state:\n        v02_be = v02_state[\'backbone_engrams\']\n        for idx_key, mod in model.backbone_engrams.items():\n            sub_dict = {\n                k.split(f\'{idx_key}.\', 1)[1]: v\n                for k, v in v02_be.items()\n                if k.startswith(f\'{idx_key}.\')\n            }\n            if sub_dict:\n                copied_eng += _copy_compatible_engram(mod, sub_dict)\n    return {\'copied_controller_tensors\': copied_ctrl, \'copied_engram_tensors\': copied_eng}\n\n'
LOADER_SOURCE = '"""One-call loader for Spark-X2.6-4B-En4B-A1B (SparkX-Loop v0.3: 8.2B Total / 5.2B Active, 72 Effective Layers)."""\nfrom __future__ import annotations\nimport json\nimport os\nfrom typing import Optional\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nfrom huggingface_hub import hf_hub_download\n\nfrom sparkx_loop_v03_core import (\n    LoopV03Settings,\n    SparkXLoopV03,\n    load_loop_v03_state_dict,\n    patch_spark_attention_for_loglinear_kv,\n)\n\n\ndef load_sparkx_loop_v03(\n    repo_or_dir: str = \'TassP0lar/Spark-X2.6-4B-En4B-A1B\',\n    base_repo: Optional[str] = None,\n    device: str = \'cuda\' if torch.cuda.is_available() else \'cpu\',\n    dtype: torch.dtype = torch.bfloat16,\n    enable_loglinear_kv: bool = False,\n    loglinear_recent_window: int = 1024,\n    loglinear_sink_tokens: int = 16,\n    offload_tier1b_cpu: bool = True,\n    token: Optional[str] = None,\n):\n    """Load a Spark-X2.6-4B-En4B-A1B (v0.3) checkpoint (72 effective layers, 8.2B total / 5.2B active)."""\n    if os.path.isdir(repo_or_dir):\n        cfg_path = os.path.join(repo_or_dir, \'loop_config.json\')\n        state_path = os.path.join(repo_or_dir, \'loop_v03_state.pt\')\n        if not os.path.exists(state_path):\n            state_path = os.path.join(repo_or_dir, \'loop_state.pt\')\n    else:\n        cfg_path = hf_hub_download(repo_or_dir, \'loop_config.json\', token=token)\n        try:\n            state_path = hf_hub_download(repo_or_dir, \'loop_v03_state.pt\', token=token)\n        except Exception:\n            state_path = hf_hub_download(repo_or_dir, \'loop_state.pt\', token=token)\n\n    with open(cfg_path, \'r\', encoding=\'utf-8\') as f:\n        meta = json.load(f)\n\n    state = torch.load(state_path, map_location=\'cpu\')\n    settings_dict = dict(state.get(\'settings\', meta.get(\'settings\', {})))\n    settings_dict[\'checkpoint_layers\'] = False\n    settings_dict[\'offload_checkpoint_to_cpu\'] = False\n    settings_dict[\'eng_offload_tier1b_cpu\'] = offload_tier1b_cpu\n    settings = LoopV03Settings(**settings_dict)\n\n    has_merged_backbone = bool(meta.get(\'merged_standalone\', True))\n    backbone_source = repo_or_dir if has_merged_backbone else (base_repo or meta.get(\'base_model\', \'XHToken/Spark-X2.6-4B\'))\n\n    tok = AutoTokenizer.from_pretrained(repo_or_dir, trust_remote_code=True, token=token)\n    base = AutoModelForCausalLM.from_pretrained(\n        backbone_source,\n        trust_remote_code=True,\n        dtype=dtype,\n        low_cpu_mem_usage=True,\n        token=token,\n    ).to(device)\n\n    model = SparkXLoopV03(base, settings, patch_sdpa=not enable_loglinear_kv).to(device)\n    load_loop_v03_state_dict(model, state, strict=False)\n\n    stage = meta.get(\'stage\', {})\n    if stage:\n        model.set_loop(\n            start=int(stage.get(\'start\', settings.start)),\n            end=int(stage.get(\'end\', settings.end)),\n            passes=int(stage.get(\'passes\', settings.passes)),\n            use_mhc=bool(stage.get(\'mhc\', settings.use_mhc)),\n            use_engram=bool(stage.get(\'engram\', settings.use_engram)),\n            enable_p3=bool(stage.get(\'enable_p3\', settings.enable_p3)),\n            p3_start=int(stage.get(\'p3_start\', settings.p3_start)),\n            p3_end=int(stage.get(\'p3_end\', settings.p3_end)),\n        )\n\n    if enable_loglinear_kv:\n        patch_spark_attention_for_loglinear_kv(\n            model.base,\n            recent_window=loglinear_recent_window,\n            sink_tokens=loglinear_sink_tokens,\n        )\n\n    model.eval()\n    return model, tok, meta\n'

Path('sparkx_loop_v03_core.py').write_text(CORE_SOURCE, encoding='utf-8')
Path('load_inference_v03.py').write_text(LOADER_SOURCE, encoding='utf-8')
print('Wrote sparkx_loop_v03_core.py and load_inference_v03.py to working directory.')

In [ ]:
# Cell 6: Instantiate Spark-X2.6-4B-En4B-A1B, Warm-Start from v0.2 (R24), and Verify Step-0 Parity
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download, HfApi
from sparkx_loop_v03_core import (
    LoopV03Settings,
    SparkXLoopV03,
    count_en4b_a1b_params,
    warm_start_from_v02_state,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

print(f"Loading tokenizer & base model from {CFG['BASE_MODEL_ID']}...")
tokenizer = AutoTokenizer.from_pretrained(CFG['BASE_MODEL_ID'], trust_remote_code=True, token=HF_TOKEN or None)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    CFG['BASE_MODEL_ID'],
    trust_remote_code=True,
    dtype=dtype,
    low_cpu_mem_usage=True,
    token=HF_TOKEN or None,
).to(device)
base_model.config.use_cache = False

# Optional Warm-Start from v0.2 (`TassP0lar/Spark-X2.6-4B-Loop-mHC-R24`)
v02_loop_state = None
if CFG.get('ENABLE_V02_WARM_START') and CFG.get('WARM_START_V02_REPO'):
    v02_repo = CFG['WARM_START_V02_REPO']
    try:
        api = HfApi(token=HF_TOKEN or None)
        repo_files = set(api.list_repo_files(v02_repo, repo_type='model'))
        if 'adapter_config.json' in repo_files:
            from peft import PeftModel
            print(f'Merging trained v0.2 LoRA adapter from {v02_repo} into BF16 backbone...')
            peft_wrapped = PeftModel.from_pretrained(base_model, v02_repo, token=HF_TOKEN or None)
            base_model = peft_wrapped.merge_and_unload()
            print('v0.2 LoRA merged cleanly into 36-layer backbone.')
        elif 'model.safetensors' in repo_files or 'model.safetensors.index.json' in repo_files:
            print(f'Loading merged v0.2 backbone weights from {v02_repo}...')
            base_model = AutoModelForCausalLM.from_pretrained(
                v02_repo,
                trust_remote_code=True,
                dtype=dtype,
                low_cpu_mem_usage=True,
                token=HF_TOKEN or None,
            ).to(device)
            base_model.config.use_cache = False
        if 'loop_state.pt' in repo_files:
            lp_path = hf_hub_download(v02_repo, 'loop_state.pt', token=HF_TOKEN or None)
            v02_loop_state = torch.load(lp_path, map_location='cpu')
            print(f'Downloaded v0.2 loop_state.pt from {v02_repo}.')
    except Exception as exc:
        print(f'v0.2 warm-start notice (continuing with clean base backbone): {exc}')

settings = LoopV03Settings(
    start=CFG['LOOP_START'],
    end=CFG['LOOP_END'],
    passes=CFG['LOOP_PASSES'],
    p3_start=CFG['P3_START'],
    p3_end=CFG['P3_END'],
    enable_p3=False,
    max_passes=CFG['MAX_PASSES'],
    mhc_streams=CFG['MHC_STREAMS'],
    eng_hot_buckets=CFG['ENG_HOT_BUCKETS'],
    eng_cold_buckets=CFG['ENG_COLD_BUCKETS'],
    eng_dim=CFG['ENG_DIM'],
    eng_shared_rank=CFG['ENG_SHARED_RANK'],
    eng_experts=CFG['ENG_EXPERTS'],
    eng_initial_active_experts=CFG['ENG_INITIAL_ACTIVE_EXPERTS'],
    eng_top_k=CFG['ENG_TOP_K'],
    eng_initial_top_k=CFG['ENG_INITIAL_TOP_K'],
    eng_expert_rank=CFG['ENG_EXPERT_RANK'],
    eng_route_dim=CFG['ENG_ROUTE_DIM'],
    eng_z_loss_coeff=CFG['ENG_Z_LOSS_COEFF'],
    eng_offload_tier1b_cpu=CFG['OFFLOAD_TIER1B_CPU'],
    backbone_engram_stride=CFG['BACKBONE_ENGRAM_STRIDE'],
    checkpoint_layers=True,
    offload_checkpoint_to_cpu=CFG.get('OFFLOAD_CHECKPOINT_CPU', False),
    pass_adapter_rank=CFG['PASS_ADAPTER_RANK'],
    ce_chunk_tokens=CFG['CE_CHUNK_TOKENS'],
)

model = SparkXLoopV03(base_model, settings=settings, strict=True, patch_sdpa=True).to(device)

# Verify exact Step-0 function-preserving parity BEFORE loading non-zero v0.2 extra_gain
model.eval()
with torch.no_grad():
    probe_ids = torch.randint(10, 5000, (1, 32), device=device)
    model.set_loop(4, 28, passes=1, use_mhc=False, use_engram=False, enable_p3=False)
    ref_logits = model.forward_logits(probe_ids)
    model.set_loop(4, 28, passes=2, use_mhc=True, use_engram=True, enable_p3=True, p3_start=12, p3_end=24)
    loop72_logits = model.forward_logits(probe_ids)
    step0_diff = (ref_logits.float() - loop72_logits.float()).abs().max().item()
print(f'Step-0 36L vs 72L Zero-Gate Parity Check: max_abs_diff = {step0_diff:.8f}')
assert step0_diff == 0.0, f'Expected exact 0.0 Step-0 parity, got {step0_diff}'

# Apply v0.2 controller warm-start if available (while keeping Pass-3 p3_extra_gain at 0.0!)
if v02_loop_state is not None:
    ws_stats = warm_start_from_v02_state(model, v02_loop_state)
    print(f'Warm-started v0.3 controllers from v0.2 state: {ws_stats}')

# Audit full parameter breakdown at full 32-expert (Top-K=7) capacity
for mod in model.all_engram_modules():
    mod.active_experts = mod.num_experts
    mod.active_top_k = mod.target_top_k
param_audit = count_en4b_a1b_params(model)
for mod in model.all_engram_modules():
    mod.active_experts = CFG['ENG_INITIAL_ACTIVE_EXPERTS']
    mod.active_top_k = CFG['ENG_INITIAL_TOP_K']

print('\n=== Spark-X2.6-4B-En4B-A1B Parameter Audit ===')
for k, v in param_audit.items():
    print(f'  {k:<28s}: {v:>14,d} ({v / 1e9:.3f}B)')
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Cell 7: 5-Pillar Dataset Streaming, Plan+Execution Filter, Anti-Slop Filter & Tool-Call Normalizer
import json
import re
from datasets import load_dataset

SLOP_PHRASES = (
    'as an ai', 'as a language model', 'delve into', 'testament to',
    'tapestry of', 'in conclusion, ', 'it is important to note that',
    'certainly! here is', 'sure! i can help you with',
)

PLAN_KEYWORDS = (
    'plan', 'architecture', 'design', 'step 1', '1.', 'approach',
    'analysis', 'let us', "let's", 'strategy', 'verify', 'constraint',
)


def _is_low_slop(text: str) -> bool:
    if not CFG.get('ENABLE_ANTI_SLOP_FILTER', True):
        return True
    low = text.lower()
    hits = sum(1 for phrase in SLOP_PHRASES if phrase in low)
    return hits <= 1


CLI_AGENT_LEAK_MARKERS = (
    'claude-projects', 'c:\\users\\', '/users/user/', 'write tool', 'issue a write tool',
    'let me check the workspace', 'let me check the directory', 'let me check the files',
    'use the bash tool', 'use the edit tool',
)


def _has_plan_and_execution(think_text: str, answer_text: str, pillar: str) -> bool:
    if not CFG.get('REQUIRE_PLAN_AND_EXECUTION', True):
        return True
    low_t = think_text.lower()
    low_a = answer_text.lower()
    if pillar != 'agentic_tool':
        if any(m in low_t or m in low_a for m in CLI_AGENT_LEAK_MARKERS):
            return False
        if len(think_text.strip()) > 400 and len(answer_text.strip()) < 260:
            return False
    if pillar in ('human_prose', 'chat'):
        return len(answer_text.strip()) >= 80 and _is_low_slop(answer_text)
    if pillar == 'agentic_tool':
        return ('<tool_call>' in think_text or '<tool_call>' in answer_text) and len(answer_text.strip()) >= 30
    if len(think_text.strip()) < 60 or len(answer_text.strip()) < 160:
        return False
    has_kw = any(kw in low_t for kw in PLAN_KEYWORDS)
    return has_kw and _is_low_slop(answer_text)


def _normalize_tool_record(row: dict) -> tuple[str, str, str] | None:
    """Convert heterogeneous tool-calling schemas (Toucan/ToolACE/xLAM/APIGen/Hermes) into unified
    <think> ... </think><tool_call>...</tool_call><tool_response>...</tool_response><think>...</think>Answer."""
    messages = row.get('messages') or row.get('conversations')
    tools = row.get('tools') or row.get('functions') or ''
    if isinstance(tools, (list, dict)):
        try:
            tools_str = json.dumps(tools, ensure_ascii=False)
        except Exception:
            tools_str = str(tools)
    else:
        tools_str = str(tools).strip()

    if isinstance(messages, list) and len(messages) >= 2:
        user_parts, assistant_parts, think_parts = [], [], []
        for m in messages:
            if not isinstance(m, dict):
                continue
            role = str(m.get('role') or m.get('from') or '').lower()
            content = str(m.get('content') or m.get('value') or '').strip()
            if not content:
                continue
            if role in ('system', 'tools') and not tools_str:
                tools_str = content
            elif role in ('user', 'human'):
                user_parts.append(content)
            elif role in ('tool', 'observation', 'function', 'environment'):
                assistant_parts.append(f'<tool_response>\n{content}\n</tool_response>')
            elif role in ('assistant', 'gpt', 'model'):
                tc = m.get('tool_calls') or m.get('function_call')
                if tc:
                    tc_json = json.dumps(tc, ensure_ascii=False) if not isinstance(tc, str) else tc
                    content = f'{content}\n<tool_call>\n{tc_json}\n</tool_call>'.strip()
                assistant_parts.append(content)
        if user_parts and assistant_parts:
            prompt = '\n\n'.join(user_parts)
            if tools_str:
                prompt = f'Available Tools:\n{tools_str[:2000]}\n\nUser Request:\n{prompt}'
            combined_a = '\n'.join(assistant_parts)
            if '<think>' in combined_a and '</think>' in combined_a:
                m_th = re.search(r'<think>(.*?)</think>', combined_a, flags=re.DOTALL)
                th = m_th.group(1).strip() if m_th else ''
                ans = re.sub(r'<think>.*?</think>', '', combined_a, count=1, flags=re.DOTALL).strip()
                return prompt, th, ans
            return prompt, 'Analyze available tools, validate parameters, and invoke the appropriate tool.', combined_a

    # Fallback for xLAM query / answers / thought format
    query = str(row.get('query') or row.get('instruction') or row.get('prompt') or '').strip()
    answers = row.get('answers') or row.get('tool_calls') or row.get('output') or row.get('response')
    thought = str(row.get('thought') or row.get('reasoning') or row.get('thinking') or '').strip()
    if query and answers:
        ans_str = json.dumps(answers, ensure_ascii=False) if isinstance(answers, (list, dict)) else str(answers).strip()
        if '<tool_call>' not in ans_str:
            ans_str = f'<tool_call>\n{ans_str}\n</tool_call>'
        if tools_str:
            query = f'Available Tools:\n{tools_str[:2000]}\n\nUser Request:\n{query}'
        if not thought:
            thought = 'Select the matching tool signature and construct verified JSON arguments.'
        return query, thought, ans_str
    return None


def _extract_standard_record(row: dict, pillar: str) -> dict | None:
    if pillar == 'agentic_tool':
        norm = _normalize_tool_record(row)
        if norm is not None:
            p, th, ans = norm
            if _has_plan_and_execution(th, ans, pillar):
                return {'prompt': p, 'think': th, 'answer': ans, 'pillar': pillar}

    prompt, think, answer = '', '', ''
    messages = row.get('messages') or row.get('conversations')
    if isinstance(messages, list) and len(messages) >= 2:
        u_list, a_list = [], []
        for m in messages:
            if not isinstance(m, dict):
                continue
            r = str(m.get('role') or m.get('from') or '').lower()
            c = str(m.get('content') or m.get('value') or '').strip()
            if r in ('user', 'human'):
                u_list.append(c)
            elif r in ('assistant', 'gpt', 'model'):
                a_list.append(c)
        if u_list and a_list:
            prompt = '\n\n'.join(u_list[:2])
            answer = '\n\n'.join(a_list[:2])
    if not prompt:
        for k in ('prompt', 'instruction', 'question', 'problem', 'input', 'query'):
            if row.get(k):
                prompt = str(row[k]).strip()
                break
    if not answer:
        for k in ('response', 'output', 'answer', 'solution', 'completion', 'text'):
            if row.get(k):
                answer = str(row[k]).strip()
                break
    for k in ('reasoning', 'thinking', 'thought', 'reasoning_content', 'cot'):
        if row.get(k):
            think = str(row[k]).strip()
            break

    if '<think>' in answer and '</think>' in answer:
        m = re.search(r'<think>(.*?)</think>', answer, flags=re.DOTALL)
        if m and not think:
            think = m.group(1).strip()
        answer = re.sub(r'<think>.*?</think>', '', answer, count=1, flags=re.DOTALL).strip()

    if not prompt or not answer:
        return None
    if not _has_plan_and_execution(think, answer, pillar):
        return None
    return {'prompt': prompt, 'think': think, 'answer': answer, 'pillar': pillar}


curated_records = []
pillar_counts = {}
for src in CFG['CURATED_SOURCES']:
    ds_id = src['id']
    pillar = src['pillar']
    cap = int(src.get('cap', 300))
    kept = 0
    for split_name in ('train', 'sft', 'default'):
        try:
            ds_stream = load_dataset(ds_id, split=split_name, streaming=True, token=HF_TOKEN or None)
            for row in ds_stream:
                rec = _extract_standard_record(row, pillar)
                if rec is not None:
                    rec['source'] = ds_id
                    curated_records.append(rec)
                    kept += 1
                    pillar_counts[pillar] = pillar_counts.get(pillar, 0) + 1
                    if kept >= cap:
                        break
            if kept > 0:
                break
        except Exception:
            continue
    print(f'  [{pillar:<12s}] {ds_id:<52s} -> kept {kept:4d} samples')

print(f'\nTotal Curated High-Signal Samples: {len(curated_records):,d} | Pillar Breakdown: {pillar_counts}')


In [ ]:
# Cell 8: OpenCode Go (`https://opencode.ai/zen/go/v1`) Teacher Verification & <think> Enrichment Pass
import json
import urllib.request


def _opencode_enrich_think(prompt_text: str, answer_text: str) -> str:
    if not OPENCODE_API_KEY:
        return ''
    url = f"{CFG['OPENCODE_BASE_URL'].rstrip('/')}/chat/completions"
    sys_msg = (
        'You are a concise architectural reasoning verifier. Given a user problem and its target response, '
        'write a compact, high-signal 3-step engineering plan (1. Problem Decomposition, 2. Edge Cases & Constraints, '
        '3. Execution Verification) in 60-120 words. Output ONLY the plan.'
    )
    payload = {
        'model': CFG.get('OPENCODE_MODEL', 'glm-5'),
        'messages': [
            {'role': 'system', 'content': sys_msg},
            {'role': 'user', 'content': f'Problem:\n{prompt_text[:1200]}\n\nTarget Output:\n{answer_text[:1200]}'},
        ],
        'max_tokens': 220,
        'temperature': 0.2,
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {OPENCODE_API_KEY}',
            'User-Agent': 'SparkX-Loop-v03/1.0',
        },
        method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            body = json.loads(resp.read().decode('utf-8'))
            return body['choices'][0]['message']['content'].strip()
    except Exception:
        return ''


enriched_count = 0
max_enrich = int(CFG.get('OPENCODE_ENRICH_SAMPLES', 24))
if OPENCODE_API_KEY and max_enrich > 0:
    for rec in curated_records:
        if enriched_count >= max_enrich:
            break
        if rec['pillar'] in ('planning', 'stem_code') and len(rec.get('think', '')) < 120:
            new_think = _opencode_enrich_think(rec['prompt'], rec['answer'])
            if len(new_think) >= 60:
                rec['think'] = new_think
                enriched_count += 1
    print(f'OpenCode Go teacher enriched {enriched_count} reasoning traces.')
else:
    print('Skipping remote OpenCode Go enrichment pass (using native dataset <think> traces).')


In [ ]:
# Cell 9: Tokenization in 130 GB System RAM with `<tool_response>` Label Masking (`labels = -100`) & Document-Aligned Packing
import random
import re
import torch

random.seed(42)
random.shuffle(curated_records)

seq_len = int(CFG['SEQ_LEN'])
eos_id = tokenizer.eos_token_id or 2


PILLAR_SYSTEM_PROMPTS = {
    'planning': (
        'You are Spark-X2.6-4B-En4B-A1B, a 72-effective-layer recursive reasoning and software architecture AI. '
        'Plan concisely inside <think>...</think> and deliver complete, un-truncated, self-contained implementations outside </think>.'
    ),
    'stem_code': (
        'You are Spark-X2.6-4B-En4B-A1B, a 72-effective-layer scientific, mathematical, and systems coding assistant. '
        'Derive and verify every step inside <think>...</think> before presenting the complete solution.'
    ),
    'agentic_tool': (
        'You are Spark-X2.6-4B-En4B-A1B, an autonomous agentic coding and tool-use model. '
        'Deliberate inside <think>...</think>, invoke tools via <tool_call>{...}</tool_call>, and integrate <tool_response> observations.'
    ),
    'human_prose': (
        'You are Spark-X2.6-4B-En4B-A1B. Write in clear, natural, vivid human prose free of AI cliches.'
    ),
    'chat': (
        'You are Spark-X2.6-4B-En4B-A1B, a helpful, accurate, and thoughtful multi-turn assistant.'
    ),
}


def _format_prompt_ids(sys_txt: str, user_txt: str) -> list[int]:
    messages = [
        {'role': 'system', 'content': sys_txt},
        {'role': 'user', 'content': user_txt},
    ]
    if hasattr(tokenizer, 'apply_chat_template') and getattr(tokenizer, 'chat_template', None):
        try:
            prompt_str = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            if prompt_str.endswith('<think>\n'):
                prompt_str = prompt_str[:-len('<think>\n')]
            elif prompt_str.endswith('<think>'):
                prompt_str = prompt_str[:-len('<think>')]
            return tokenizer.encode(prompt_str, add_special_tokens=False)
        except Exception:
            pass
    prompt_str = f'<|im_start|>system\n{sys_txt}<|im_end|>\n<|im_start|>user\n{user_txt}<|im_end|>\n<|im_start|>assistant\n'
    return tokenizer.encode(prompt_str, add_special_tokens=False)


def _encode_record_with_masks(rec: dict) -> tuple[list[int], list[int]]:
    """Encode record with native Chat Template, <tool_response> masking (-100), and zero mid-thought truncation."""
    sys_txt = PILLAR_SYSTEM_PROMPTS.get(rec.get('pillar', 'planning'), PILLAR_SYSTEM_PROMPTS['planning'])
    prompt_ids = _format_prompt_ids(sys_txt, rec['prompt'].strip())
    input_ids = list(prompt_ids)
    labels = [-100] * len(prompt_ids)

    think_txt = rec.get('think', '').strip()
    ans_txt = rec['answer'].strip()
    if think_txt:
        # Keep whole sentences if <think> is very long; never slice mid-word or glue </think> to a cut token
        if len(think_txt) > 1800:
            sentences = re.split(r'(?<=[.!?\n])\s+', think_txt)
            kept = []
            cur_chars = 0
            for s in sentences:
                if cur_chars + len(s) > 1600 and kept:
                    break
                kept.append(s)
                cur_chars += len(s) + 1
            think_txt = ' '.join(kept).strip()
        full_assistant = f'<think>\n{think_txt}\n</think>\n{ans_txt}'
    else:
        full_assistant = ans_txt

    if CFG.get('MASK_TOOL_RESPONSE_LABELS', True) and '<tool_response>' in full_assistant:
        parts = re.split(r'(<tool_response>.*?</tool_response>)', full_assistant, flags=re.DOTALL)
        for part in parts:
            if not part:
                continue
            p_ids = tokenizer.encode(part, add_special_tokens=False)
            input_ids.extend(p_ids)
            if part.startswith('<tool_response>'):
                labels.extend([-100] * len(p_ids))
            else:
                labels.extend(p_ids)
    else:
        a_ids = tokenizer.encode(full_assistant, add_special_tokens=False)
        input_ids.extend(a_ids)
        labels.extend(a_ids)

    input_ids.append(eos_id)
    labels.append(eos_id)
    return input_ids, labels


packed_inputs, packed_labels = [], []
buf_in, buf_lbl = [], []
skipped_overlong = 0

for rec in curated_records:
    r_in, r_lbl = _encode_record_with_masks(rec)
    if sum(1 for x in r_lbl if x != -100) < 16:
        continue
    # Never truncate a document mid-answer or split a single document across chunks
    if len(r_in) > seq_len:
        skipped_overlong += 1
        continue
    if len(buf_in) + len(r_in) > seq_len and buf_in:
        pad_len = seq_len - len(buf_in)
        chunk_in = buf_in + [eos_id] * pad_len
        chunk_lbl = buf_lbl + [-100] * pad_len
        if any(x != -100 for x in chunk_lbl[1:]):
            packed_inputs.append(torch.tensor(chunk_in, dtype=torch.long))
            packed_labels.append(torch.tensor(chunk_lbl, dtype=torch.long))
        buf_in, buf_lbl = [], []
    buf_in.extend(r_in)
    buf_lbl.extend(r_lbl)

if buf_in and any(x != -100 for x in buf_lbl[1:]):
    pad_len = seq_len - len(buf_in)
    packed_inputs.append(torch.tensor(buf_in + [eos_id] * pad_len, dtype=torch.long))
    packed_labels.append(torch.tensor(buf_lbl + [-100] * pad_len, dtype=torch.long))

if not packed_inputs:
    raise RuntimeError('No packed sequences produced; check dataset streaming connectivity.')

val_count = max(4, min(24, len(packed_inputs) // 20))
val_inputs = torch.stack(packed_inputs[:val_count])
val_labels = torch.stack(packed_labels[:val_count])
train_inputs = torch.stack(packed_inputs[val_count:])
train_labels = torch.stack(packed_labels[val_count:])

print(
    f'Document-Aligned Packed Dataset Ready in System RAM: '
    f'Train = {train_inputs.shape[0]:,d} sequences x {seq_len} tokens '
    f'({train_inputs.numel() / 1e6:.2f}M tokens) | Val = {val_inputs.shape[0]} sequences '
    f'(skipped_overlong={skipped_overlong})'
)

In [ ]:
# Cell 10: Rolling Max-3 Checkpoint Exporter & Hugging Face Hub Uploader (`TassP0lar/Spark-X2.6-4B-En4B-A1B`)
import json
import os
import shutil
from pathlib import Path
import torch
from huggingface_hub import HfApi
from sparkx_loop_v03_core import count_en4b_a1b_params, loop_v03_state_dict

saved_stage_dirs: list[Path] = []


def export_stage_v03(stage_cfg: dict, global_step: int, val_loss: float, push_hub: bool = False):
    out_root = Path(CFG['OUTPUT_DIR'])
    out_root.mkdir(parents=True, exist_ok=True)
    stage_name = stage_cfg['name']
    ckpt_dir = out_root / f'stage_{stage_name}_step_{global_step}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n[Export] Saving standalone v0.3 checkpoint to {ckpt_dir}...')
    model.base.save_pretrained(ckpt_dir, safe_serialization=True, max_shard_size='5GB')
    tokenizer.save_pretrained(ckpt_dir)

    state = loop_v03_state_dict(model, include_tier1b=True)
    torch.save(state, ckpt_dir / 'loop_v03_state.pt')

    p_summary = count_en4b_a1b_params(model)
    loop_meta = {
        'architecture': 'Spark-X2.6-4B-En4B-A1B',
        'version': 'v0.3',
        'base_model': CFG['BASE_MODEL_ID'],
        'warm_start_v02_repo': CFG.get('WARM_START_V02_REPO', ''),
        'merged_standalone': True,
        'effective_virtual_layers': model.total_virtual_layers,
        'stage': stage_cfg,
        'global_step': global_step,
        'val_loss': float(val_loss),
        'parameter_summary': p_summary,
        'settings': state['settings'],
        'active_meta': state['active_meta'],
    }
    (ckpt_dir / 'loop_config.json').write_text(json.dumps(loop_meta, indent=2), encoding='utf-8')

    for helper_file in ('sparkx_loop_v03_core.py', 'load_inference_v03.py'):
        if Path(helper_file).exists():
            shutil.copy2(helper_file, ckpt_dir / helper_file)

    saved_stage_dirs.append(ckpt_dir)
    max_keep = int(CFG.get('MAX_SAVED_CHECKPOINTS', 3))
    while len(saved_stage_dirs) > max_keep:
        old_dir = saved_stage_dirs.pop(0)
        if old_dir.exists() and old_dir != ckpt_dir:
            shutil.rmtree(old_dir, ignore_errors=True)
            print(f'[Disk Guard] Pruned older local checkpoint {old_dir.name}')

    if push_hub and CFG.get('PUSH_TO_HUB') and HF_TOKEN:
        repo_id = CFG['HF_HUB_REPO_ID']
        try:
            api = HfApi(token=HF_TOKEN)
            api.create_repo(repo_id=repo_id, repo_type='model', private=CFG.get('PRIVATE_HUB_REPO', False), exist_ok=True)
            api.upload_folder(
                repo_id=repo_id,
                folder_path=str(ckpt_dir),
                repo_type='model',
                commit_message=f'Spark-X2.6-4B-En4B-A1B stage={stage_name} ({model.total_virtual_layers}L) step={global_step} val_loss={val_loss:.4f}',
            )
            print(f'[Hub Push] Uploaded {stage_name} ({model.total_virtual_layers} virtual layers) -> https://huggingface.co/{repo_id}')
        except Exception as exc:
            print(f'[Hub Push Warning] Failed to upload stage {stage_name}: {exc}')
    return ckpt_dir


In [ ]:
# Cell 11: 4-Stage Progressive Curriculum (`warmup [36L] -> R12 [48L] -> R24 [60L, 8->32 Upcycle] -> R72 [72L]`)
import math
import time
import torch
from sparkx_loop_v03_core import build_optimizer_v03, set_trainables_v03


@torch.no_grad()
def evaluate_val_loss() -> float:
    was_training = model.training
    model.eval()
    losses = []
    n_eval = min(8, val_inputs.shape[0])
    for i in range(n_eval):
        xb = val_inputs[i : i + 1].to(device, non_blocking=True)
        yb = val_labels[i : i + 1].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=dtype, enabled=(device == 'cuda')):
            out = model(xb, labels=yb)
        losses.append(float(out['ce_loss'].item()))
    model.train(was_training)
    return sum(losses) / max(1, len(losses))


global_step = 0
stage_history = []
micro_batch = int(CFG['MICRO_BATCH'])
grad_accum = int(CFG['GRAD_ACCUM'])
n_train = train_inputs.shape[0]
cursor = 0

initial_val = evaluate_val_loss()
print(f'Initial Validation CE Loss before v0.3 training: {initial_val:.4f}')

for stage_idx, stage in enumerate(CFG['STAGES']):
    stage_name = stage['name']
    if stage.get('upcycle_32', False):
        print(f'\n>>> [Coarse-to-Fine Upcycling] Cloning & orthogonally perturbing 8 -> 32 Routed Experts (Top-K={CFG["ENG_TOP_K"]})...')
        model.upcycle_all_engrams_8_to_32(noise_std=1e-3)

    model.set_loop(
        start=int(stage['start']),
        end=int(stage['end']),
        passes=int(stage['passes']),
        use_mhc=bool(stage['mhc']),
        use_engram=bool(stage['engram']),
        enable_p3=bool(stage.get('enable_p3', False)),
        p3_start=int(stage.get('p3_start', 12)),
        p3_end=int(stage.get('p3_end', 24)),
    )

    trainables = set_trainables_v03(
        model,
        full_finetune=bool(CFG['FULL_FINETUNE']),
        freeze_backbone=bool(stage.get('freeze_backbone', False)),
    )
    optimizer = build_optimizer_v03(
        model,
        lr_adapter=float(CFG['LR_ADAPTER']),
        lr_full=float(CFG['LR_FULL']),
        lr_mhc=float(CFG['LR_MHC']),
        weight_decay=float(CFG['WEIGHT_DECAY']),
        use_8bit=bool(CFG['USE_8BIT_OPTIM']),
    )

    stage_steps = int(stage['steps'])
    print(
        f"\n================================================================================\n"
        f"Stage {stage_idx + 1}/{len(CFG['STAGES'])}: {stage_name} | "
        f"Effective Depth = {model.total_virtual_layers} Virtual Layers | "
        f"Active Experts/Anchor = {model.engram.active_experts} (Top-K={model.engram.active_top_k})\n"
        f"Description: {stage['desc']}\n"
        f"Trainable GPU Tensors: {len(trainables)} | Steps: {stage_steps}\n"
        f"================================================================================"
    )

    model.train()
    t0 = time.time()
    for step_in_stage in range(1, stage_steps + 1):
        optimizer.zero_grad(set_to_none=True)
        accum_ce = 0.0
        accum_z = 0.0

        # Cosine LR schedule with 10% warmup per stage
        progress = step_in_stage / float(max(1, stage_steps))
        lr_mult = (progress / 0.1) if progress < 0.1 else (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * (progress - 0.1) / 0.9)))
        for pg_idx, pg in enumerate(optimizer.param_groups):
            base_lr = CFG['LR_ADAPTER'] if pg_idx == 0 else (CFG['LR_MHC'] if pg_idx == 1 else CFG['LR_FULL'])
            pg['lr'] = base_lr * lr_mult

        for _ in range(grad_accum):
            if cursor + micro_batch > n_train:
                perm = torch.randperm(n_train)
                train_inputs = train_inputs[perm]
                train_labels = train_labels[perm]
                cursor = 0
            xb = train_inputs[cursor : cursor + micro_batch].to(device, non_blocking=True)
            yb = train_labels[cursor : cursor + micro_batch].to(device, non_blocking=True)
            cursor += micro_batch

            with torch.amp.autocast('cuda', dtype=dtype, enabled=(device == 'cuda')):
                out = model(xb, labels=yb)
                loss = out['loss'] / float(grad_accum)

            loss.backward()
            # Coalesced sparse Adam step for offloaded Host DRAM Tier-1B rows (<0.3 ms)
            model.step_all_tier1b_sparse_adam(lr=float(CFG['LR_ADAPTER']) * lr_mult)
            accum_ce += float(out['ce_loss'].item()) / float(grad_accum)
            accum_z += float(out['z_loss'].item() if torch.is_tensor(out['z_loss']) else out['z_loss']) / float(grad_accum)

        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(CFG['GRAD_CLIP']))
        optimizer.step()
        global_step += 1

        if step_in_stage == 1 or step_in_stage % 10 == 0 or step_in_stage == stage_steps:
            elapsed = time.time() - t0
            ctrl_sample = model.controllers[str(model.loop_start)]
            p2_gate = math.tanh(float(ctrl_sample.extra_gain.item()))
            p3_gate = math.tanh(float(ctrl_sample.p3_extra_gain.item()))
            eng_gate = 1.0 / (1.0 + math.exp(-float(model.engram.gain.item())))
            print(
                f"[{stage_name} {step_in_stage:3d}/{stage_steps:3d} | step {global_step:4d}] "
                f"CE={accum_ce:.4f} | Z={accum_z:.5f} | "
                f"p2_tanh={p2_gate:+.4f} | p3_tanh={p3_gate:+.4f} | eng_sig={eng_gate:.4f} | "
                f"{elapsed:.1f}s"
            )

    val_ce = evaluate_val_loss()
    stage_history.append({
        'stage': stage_name,
        'virtual_layers': model.total_virtual_layers,
        'active_experts': model.engram.active_experts,
        'active_top_k': model.engram.active_top_k,
        'global_step': global_step,
        'val_ce_loss': val_ce,
    })
    print(f'--> Stage {stage_name} ({model.total_virtual_layers}L) Validation CE Loss: {val_ce:.4f}')
    is_final_stage = (stage_idx == len(CFG['STAGES']) - 1)
    export_stage_v03(stage, global_step=global_step, val_loss=val_ce, push_hub=is_final_stage)


In [ ]:
# Cell 12: 72-Layer Verification, Runtime O(n log n) Log-Linear KV Cache & 3D WebGL Voxel Cherry Blossom Garden Test
import html
import re
import time
import torch
from IPython.display import HTML, display
from sparkx_loop_v03_core import patch_spark_attention_for_loglinear_kv

model.eval()

# 1. Enable Runtime-Only Zero-Collapse O(n log n) Hierarchical Fenwick Log-Linear KV Cache on NoPE Layers
patched_layers = patch_spark_attention_for_loglinear_kv(
    model.base,
    recent_window=1024,
    sink_tokens=16,
)
print(f'Patched {patched_layers} Spark layers with Runtime O(n log n) Log-Linear KV Cache for NoPE layers.')

# 2. Quick 72-Layer Reasoning & Tool-Call Sanity Check
test_prompt = (
    f"<|system|>\n{PILLAR_SYSTEM_PROMPTS['agentic_tool']}\n"
    '<|user|>\nAvailable Tools:\n'
    '[{"name": "query_cluster_metrics", "parameters": {"cluster_id": "string", "window_mins": "integer"}}]\n\n'
    'User Request:\nCheck the p99 latency on cluster `us-east-prod-04` over the last 15 minutes.\n<|assistant|>\n'
)
inp_ids = tokenizer(test_prompt, return_tensors='pt').input_ids.to(device)
with torch.no_grad(), torch.amp.autocast('cuda', dtype=dtype, enabled=(device == 'cuda')):
    gen_ids = model.generate_cached(inp_ids, max_new_tokens=180, eos_token_id=tokenizer.eos_token_id)
gen_text = tokenizer.decode(gen_ids[0][inp_ids.shape[1]:], skip_special_tokens=True)
print('\n=== 72-Layer Agentic Tool-Call Sample ===\n' + gen_text)

# 3. Comprehensive 3D WebGL Voxel Cherry Blossom (Sakura) Garden Stress Test
sakura_prompt = (
    f"<|system|>\n{PILLAR_SYSTEM_PROMPTS['planning']}\n"
    '<|user|>\nBuild a complete, self-contained, single-file HTML5 + Three.js 3D Interactive Voxel Cherry Blossom '
    '(Sakura) Garden webpage (`<!DOCTYPE html>` ... `</html>`).\n'
    'First plan the 3D scene architecture inside `<think>...</think>`, then output the complete runnable HTML file '
    'inside a ```html ... ``` code block.\n'
    'Requirements:\n'
    '1. Load Three.js (`https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js`).\n'
    '2. Use `THREE.InstancedMesh` to render a stepped voxel island terrain with a reflective water pond and wooden arched bridge.\n'
    '3. Build multi-tiered voxel Sakura (cherry blossom) trees with pink/blush canopy clusters and Japanese stone lanterns (`Toro`) with warm point lights.\n'
    '4. Animate at least 400 falling pink cherry blossom petals drifting in the wind and floating on the water.\n'
    '5. Include interactive Orbit/Drag camera controls + an on-screen HUD toggle for Day / Golden Sunset / Night Lantern mode.\n'
    '<|assistant|>\n'
)
sakura_ids = tokenizer(sakura_prompt, return_tensors='pt').input_ids.to(device)
t0 = time.time()
with torch.no_grad(), torch.amp.autocast('cuda', dtype=dtype, enabled=(device == 'cuda')):
    sakura_out_ids = model.generate_cached(sakura_ids, max_new_tokens=262144, eos_token_id=tokenizer.eos_token_id)
sakura_elapsed = time.time() - t0
sakura_response = tokenizer.decode(sakura_out_ids[0][sakura_ids.shape[1]:], skip_special_tokens=True)
new_toks = sakura_out_ids.shape[1] - sakura_ids.shape[1]
print(f'\n=== 3D Voxel Cherry Blossom Generation ({new_toks} tokens in {sakura_elapsed:.1f}s = {new_toks / max(0.1, sakura_elapsed):.1f} tok/s) ===')
print(sakura_response[:1000] + ('\n... [truncated preview] ...' if len(sakura_response) > 1000 else ''))

html_match = re.search(r'```html\s*(<!DOCTYPE html>.*?</html>)\s*```', sakura_response, flags=re.DOTALL | re.IGNORECASE)
if not html_match:
    html_match = re.search(r'(<!DOCTYPE html>.*?</html>)', sakura_response, flags=re.DOTALL | re.IGNORECASE)

if html_match:
    extracted_html = html_match.group(1)
    out_html_path = 'voxel_sakura_garden_v03_preview.html'
    with open(out_html_path, 'w', encoding='utf-8') as f:
        f.write(extracted_html)
    print(f'Saved generated 3D Voxel Cherry Blossom Garden to {out_html_path} — rendering interactive preview below:')
    escaped = html.escape(extracted_html, quote=True)
    display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="560" style="border:1px solid #f48fb1; border-radius:10px;"></iframe>'))
